In [41]:
import dspy 
from config.settings import settings


In [70]:

settings.model_dump()

{'llm_model': 'openai/gemma-4-31B-it',
 'llm_api_base': 'http://192.168.1.204:16663/v1',
 'llm_api_key': 'Oj_63y_gBe8qlO4IDb0cR0w+FsrsOEr7bOCkPqxNrZ<Pg+sreD',
 'llm_timeout': 240,
 'llm_retry_attempts': 5,
 'llm_max_concurrent': 50}

In [71]:

lm = dspy.LM(
    model=settings.llm_model,
    api_base="http://192.168.1.204:16662/v1",
    api_key=settings.llm_api_key,
    timeout=settings.llm_timeout,
)
dspy.settings.configure(lm=lm, temperature=0.0)


In [49]:
from dspy import ChainOfThought


In [149]:
import pandas as pd

dsm_df = pd.read_json("books/dsm-tree/dsm5-final.jsonl", lines=True)
dsm_df

,page_number,content,book_name,tree,llm
0,26,### **An Uncertain Diagnosis**\n\n### When you...,DSM5_ME,"{'type': 'root', 'heading': None, 'content': '...",openai/gemma-4-31B-it
1,2,### **Also from James Morrison**\n\n### Diagno...,DSM5_ME,"{'type': 'root', 'heading': None, 'content': '...",openai/gemma-4-31B-it
2,3,# **DSM-5-TR** **®**\n\n# **Made Easy**\n\n# *...,DSM5_ME,"{'type': 'root', 'heading': None, 'content': '...",openai/gemma-4-31B-it
3,4,EPUB Edition ISBN: 9781462551361\n\nCopyright ...,DSM5_ME,"{'type': 'root', 'heading': None, 'content': '...",openai/gemma-4-31B-it
4,5,"### *For Mary, always my sine qua non*",DSM5_ME,"{'type': 'root', 'heading': None, 'content': '...",openai/gemma-4-31B-it
...,...,...,...,...,...
1994,1368,"of prolonged grief disorder, 324 of PTSD, 311 ...",DSM5_TR,"{'type': 'root', 'heading': None, 'content': '...",openai/gemma-4-31B-it
1995,1343,"Kleptomania, 539 – 541 Korea, prevalence of\n\...",DSM5_TR,"{'type': 'root', 'heading': None, 'content': '...",openai/gemma-4-31B-it
1996,1373,"Systemic racism, 17 – 18\n\nTachypnea, and neu...",DSM5_TR,"{'type': 'root', 'heading': None, 'content': '...",openai/gemma-4-31B-it
1997,1320,"Cannabis, 129 , 147 , 159 , 471 , 503 . *See a...",DSM5_TR,"{'type': 'root', 'heading': None, 'content': '...",openai/gemma-4-31B-it


In [150]:
dsm_df[dsm_df['page_number']==36]

,page_number,content,book_name,tree,llm
51,36,"**Disorders of Eating, Sleeping, and Eliminati...",DSM5_ME,"{'type': 'root', 'heading': 'Disorders of Eati...",openai/gemma-4-31B-it
657,36,"as a DSM-5-TR disorder, because of its additio...",DSM5_TR,"{'type': 'root', 'heading': None, 'content': '...",openai/gemma-4-31B-it


In [72]:
import pandas as pd

def find_missing_pages(df, max_num, page_col='page_number'):
    """
    Find page numbers that are missing from a DataFrame column within the range 1..max_num.
    
    Parameters:
    - df: pandas DataFrame
    - max_num: int, the maximum expected page number (inclusive)
    - page_col: str, name of the column containing page numbers (default 'page_number')
    
    Returns:
    - list of integers representing missing page numbers
    """
    # Ensure the column exists
    if page_col not in df.columns:
        raise ValueError(f"Column '{page_col}' not found in DataFrame")
    
    # Get unique page numbers that exist (drop NaN and convert to int if needed)
    existing_pages = set(df[page_col].dropna().unique())
    
    # Full expected set of page numbers from 1 to max_num
    expected_pages = set(range(1, max_num + 1))
    
    # Missing pages = expected - existing
    missing = sorted(expected_pages - existing_pages)
    
    return missing

In [31]:
pages = pd.read_csv('books/dsm5_all.csv')
pages[pages['page_number']==37]

,Unnamed: 0,page_number,markdown,book_name
36,36,37,Individuals with intellectual developmental di...,DSM5_ME
669,36,37,xxiii\n\n## **Preface to DSM-5**\n\n### The Am...,DSM5_TR


In [153]:
find_missing_pages(dsm_df[dsm_df['book_name']=='DSM5_ME'], 633, 'page_number')

[1, 35, 241, 427, 611]

In [151]:
me_missings = pages[(pages['book_name']=='DSM5_ME') & (pages['page_number'].isin(find_missing_pages(dsm_df[dsm_df['book_name']=='DSM5_ME'], 633,'page_number')))]
me_missings

,Unnamed: 0,page_number,markdown,book_name
0,0,1,NaN,DSM5_ME
34,34,35,**Communication and Learning Disorders**\n\n**...,DSM5_ME
240,240,241,## **Chapter 8**\n\n# **Somatic Symptom and Re...,DSM5_ME
426,426,427,The characteristics of tobacco use disorder ar...,DSM5_ME
610,610,611,"## C\n\nCaffeine disorders, 424 – 428\n\nintox...",DSM5_ME


In [37]:
tr_missings = pages[(pages['book_name']=='DSM5_TR') & (pages['page_number'].isin(find_missing_pages(dsm_df[~(dsm_df['book_name']=='DSM5_ME')], 1377,'page_number')))]
tr_missings

,Unnamed: 0,page_number,markdown,book_name
633,0,1,NaN,DSM5_TR
634,1,2,NaN,DSM5_TR
783,150,151,**Sensory deficits.**\n\n**Normal speech dysfl...,DSM5_TR
1089,456,457,"conversations, activities, objects, situations...",DSM5_TR
1831,1198,1199,## **Alphabetical Listing of DSM-5-TR**\n\n## ...,DSM5_TR
1973,1340,1341,Infections. *See also* Hepatitis; HIV; Strepto...,DSM5_TR


In [73]:
missings = pd.concat([tr_missings, me_missings])
missings = missings[missings['markdown'].notna()]
missings

,Unnamed: 0,page_number,markdown,book_name
783,150,151,**Sensory deficits.**\n\n**Normal speech dysfl...,DSM5_TR
1089,456,457,"conversations, activities, objects, situations...",DSM5_TR
1831,1198,1199,## **Alphabetical Listing of DSM-5-TR**\n\n## ...,DSM5_TR
1973,1340,1341,Infections. *See also* Hepatitis; HIV; Strepto...,DSM5_TR
34,34,35,**Communication and Learning Disorders**\n\n**...,DSM5_ME
240,240,241,## **Chapter 8**\n\n# **Somatic Symptom and Re...,DSM5_ME
426,426,427,The characteristics of tobacco use disorder ar...,DSM5_ME
610,610,611,"## C\n\nCaffeine disorders, 424 – 428\n\nintox...",DSM5_ME


In [74]:
from signatures import text_tree_extraction
from dspy import ChainOfThought
cot_module = ChainOfThought(text_tree_extraction.ExtractPageTree)


In [155]:
results

[{'type': 'root',
  'heading': None,
  'content': '',
  'children': [{'type': 'heading',
    'heading': 'Sensory deficits.',
    'content': 'Sensory deficits.',
    'children': [{'type': 'paragraph',
      'heading': None,
      'content': 'Dysfluencies of speech may be associated with a hearing impairment or other sensory deficit or a speech-motor deficit. When the speech dysfluencies are in excess of those usually associated with these problems, a diagnosis of childhood-onset fluency disorder may be made.',
      'children': []}]},
   {'type': 'heading',
    'heading': 'Normal speech dysfluencies.',
    'content': 'Normal speech dysfluencies.',
    'children': [{'type': 'paragraph',
      'heading': None,
      'content': 'The disorder must be distinguished from normal dysfluencies that occur frequently in young children, which include whole-word or phrase repetitions (e.g., “I want, I want ice cream”), incomplete phrases, interjections, unfilled pauses, and parenthetical remarks. If

In [75]:
missings['tree'] = None
results = []
for i, row in missings.iterrows():
    result = cot_module(page=row['markdown'])
    results.append(result.tree)

results

[{'type': 'root',
  'heading': None,
  'content': '',
  'children': [{'type': 'heading',
    'heading': 'Sensory deficits.',
    'content': 'Sensory deficits.',
    'children': [{'type': 'paragraph',
      'heading': None,
      'content': 'Dysfluencies of speech may be associated with a hearing impairment or other sensory deficit or a speech-motor deficit. When the speech dysfluencies are in excess of those usually associated with these problems, a diagnosis of childhood-onset fluency disorder may be made.',
      'children': []}]},
   {'type': 'heading',
    'heading': 'Normal speech dysfluencies.',
    'content': 'Normal speech dysfluencies.',
    'children': [{'type': 'paragraph',
      'heading': None,
      'content': 'The disorder must be distinguished from normal dysfluencies that occur frequently in young children, which include whole-word or phrase repetitions (e.g., “I want, I want ice cream”), incomplete phrases, interjections, unfilled pauses, and parenthetical remarks. If

In [158]:
missings

,Unnamed: 0,page_number,markdown,book_name,tree
783,150,151,**Sensory deficits.**\n\n**Normal speech dysfl...,DSM5_TR,"{'type': 'root', 'heading': None, 'content': '..."
1089,456,457,"conversations, activities, objects, situations...",DSM5_TR,"{'type': 'root', 'heading': None, 'content': '..."
34,34,35,**Communication and Learning Disorders**\n\n**...,DSM5_ME,"{'type': 'root', 'heading': None, 'content': '..."
240,240,241,## **Chapter 8**\n\n# **Somatic Symptom and Re...,DSM5_ME,"{'type': 'root', 'heading': None, 'content': '..."
426,426,427,The characteristics of tobacco use disorder ar...,DSM5_ME,"{'type': 'root', 'heading': None, 'content': '..."


In [78]:
missings = missings[~(missings['page_number'].isin([611, 1199, 1341]))]
missings

,Unnamed: 0,page_number,markdown,book_name,tree
783,150,151,**Sensory deficits.**\n\n**Normal speech dysfl...,DSM5_TR,"{'type': 'root', 'heading': None, 'content': '..."
1089,456,457,"conversations, activities, objects, situations...",DSM5_TR,"{'type': 'root', 'heading': None, 'content': '..."
34,34,35,**Communication and Learning Disorders**\n\n**...,DSM5_ME,"{'type': 'root', 'heading': None, 'content': '..."
240,240,241,## **Chapter 8**\n\n# **Somatic Symptom and Re...,DSM5_ME,"{'type': 'root', 'heading': None, 'content': '..."
426,426,427,The characteristics of tobacco use disorder ar...,DSM5_ME,"{'type': 'root', 'heading': None, 'content': '..."


In [69]:
results[0]['children'][10]

{'type': 'paragraph',
 'heading': 'None',
 'content': 'Children who have dysfluencies when they read aloud may be diagnosed mistakenly as having a reading disorder. Oral reading fluency typically is measured by timed assessments. Slower reading rates may not accurately reflect the actual reading ability of children who stutter.',
 'children': []}

In [50]:
result = cot_module(page=missings['markdown'].tolist()[0])
result

Prediction(
    reasoning='The input page was parsed into a hierarchical structure reflecting its visual and semantic layout. Bold standalone phrases at the top were treated as individual paragraph nodes, as they appear to be a list of differential diagnoses or exclusion criteria. The ICD code `**F80.82**` was kept as a separate paragraph. Subsequent explanatory text blocks were extracted as distinct `paragraph` nodes, preserving exact spacing and punctuation. The `**Comorbidity**` section was structured as a `heading` node with its continuing text (including the embedded page number `54`) as a child `paragraph`. The final section begins with `### **Social (Pragmatic) Communication Disorder**` as a parent heading, containing `**Diagnostic Criteria**` as a child heading, which in turn holds the criterion text as a `paragraph`. All `content` and `heading` strings are exact, unaltered substrings from the input, including the incidental prompt text that appeared at the very end of the page

In [22]:
dsm_df = dsm_df.drop_duplicates(['page_number', 'book_name'], keep='first')
dsm_df

,page_number,content,book_name,tree,llm
0,26,### **An Uncertain Diagnosis**\n\n### When you...,DSM5_ME,"{'type': 'root', 'heading': None, 'content': '...",openai/gemma-4-31B-it
1,2,### **Also from James Morrison**\n\n### Diagno...,DSM5_ME,"{'type': 'root', 'heading': None, 'content': '...",openai/gemma-4-31B-it
2,3,# **DSM-5-TR** **®**\n\n# **Made Easy**\n\n# *...,DSM5_ME,"{'type': 'root', 'heading': None, 'content': '...",openai/gemma-4-31B-it
3,4,EPUB Edition ISBN: 9781462551361\n\nCopyright ...,DSM5_ME,"{'type': 'root', 'heading': None, 'content': '...",openai/gemma-4-31B-it
4,5,"### *For Mary, always my sine qua non*",DSM5_ME,"{'type': 'root', 'heading': None, 'content': '...",openai/gemma-4-31B-it
...,...,...,...,...,...
2213,1368,"of prolonged grief disorder, 324 of PTSD, 311 ...",DSM5_TR,"{'type': 'root', 'heading': None, 'content': '...",openai/gemma-4-31B-it
2214,1343,"Kleptomania, 539 – 541 Korea, prevalence of\n\...",DSM5_TR,"{'type': 'root', 'heading': None, 'content': '...",openai/gemma-4-31B-it
2215,1373,"Systemic racism, 17 – 18\n\nTachypnea, and neu...",DSM5_TR,"{'type': 'root', 'heading': None, 'content': '...",openai/gemma-4-31B-it
2216,1320,"Cannabis, 129 , 147 , 159 , 471 , 503 . *See a...",DSM5_TR,"{'type': 'root', 'heading': None, 'content': '...",openai/gemma-4-31B-it


In [20]:
dsm_df.to_json('books/dsm-tree/dsm5-final.jsonl', orient="records", lines=True)


In [79]:
import pandas as pd
import uuid

# Example tree (your input)
page_tree = {
    'type': 'root',
    'heading': None,
    'content': '',
    'children': [
        {'type': 'paragraph', 'heading': None, 'content': 'OK, so these pointers aren’t exactly iron-clad. Remember, they’re straws, not steel.', 'children': []},
        {'type': 'heading', 'heading': 'Essential Features of Psychotic Disorder Due to Another', 'content': 'Essential Features of Psychotic Disorder Due to Another', 'children': [
            {'type': 'heading', 'heading': 'Medical Condition', 'content': 'Medical Condition', 'children': [
                {'type': 'paragraph', 'heading': None, 'content': 'Through physiological means, a medical condition appears to have caused an illness that features obvious hallucinations or delusions.', 'children': []},
                {'type': 'heading', 'heading': 'The Fine Print', 'content': 'The Fine Print', 'children': [
                    {'type': 'paragraph', 'heading': None, 'content': 'For pointers on deciding when a physical condition may have caused a mental disorder, see the sidebar above .', 'children': []}
                ]},
                {'type': 'paragraph', 'heading': None, 'content': 'The D’s: • Distress or disability (work/academic, social, or personal impairment) • Differential diagnosis ( delirium, another mental disorder, substance-induced psychotic disorder, schizophrenia and its cousins, delusional disorder)', 'children': []},
                {'type': 'heading', 'heading': 'Coding Notes', 'content': 'Coding Notes', 'children': [
                    {'type': 'paragraph', 'heading': None, 'content': 'In recording the diagnosis, use the name of the responsible medical condition, and list first the medical condition, with its code number.', 'children': []},
                    {'type': 'paragraph', 'heading': None, 'content': 'Code, based on the predominant symptoms:\n\nF06.2 With delusions F06.0 With hallucinations', 'children': []},
                    {'type': 'paragraph', 'heading': None, 'content': 'You may specify severity, though you don’t have to ( p. 74 ).', 'children': []}
                ]}
            ]}
        ]},
        {'type': 'case_study', 'heading': 'Rodrigo Chavez', 'content': 'Rodrigo Chavez', 'children': [
            {'type': 'paragraph', 'heading': None, 'content': 'Since retiring from teaching at age 65, Rodrigo Chavez spends most of his time sitting alone in his room. Sometimes he plays the acoustic guitar; once or twice he’s shot targets at the rifle range. True to his lifelong custom, he never drinks. Other than his immediate family, he has few social contacts. “My cigarettes are my best friends,” he says during the forensic examination.', 'children': []},
            {'type': 'paragraph', 'heading': None, 'content': 'When Rodrigo is nearly 70, an inoperable carcinoma of the lung is diagnosed. After a course of palliative radiotherapy, he declines further treatment and settles down in his apartment to die. Four months later, he notices right-sided headaches that will sometimes awaken him in the middle of the night. Because the doctors have told him he is terminally ill, he doesn’t seek further medical attention.', 'children': []},
            {'type': 'paragraph', 'heading': None, 'content': 'Then, he begins to associate the headaches with natural gas, which he can smell coming out of the ventilator duct in his bathroom. When he calls to report the problem to Mrs. Riordan, his landlady, she sends around the building’s handyman, who can find nothing wrong. But both the odors and his headaches have increased; Rodrigo recalls that, weeks earlier, Mrs. Riordan went out several times to watch while', 'children': []}
        ]}
    ]
}

#  Recursive function to assign unique IDs and mark leaf/parent
def assign_ids(node, book_name="DSM5", parent_id=None, page_number=1, nodes_list=None):
    if nodes_list is None:
        nodes_list = []

    node_id = str(uuid.uuid4())  # unique ID
    # Determine if node is a leaf or has children
    node_type_in_tree = 'leaf' if not node.get('children') else 'parent'

    node_record = {
        'node_id': node_id,
        'parent_id': parent_id,
        'type': node.get('type'),
        'heading': node.get('heading'),
        'content': node.get('content'),
        'page': page_number,
        'node_type_in_tree': node_type_in_tree,
        'book_name': book_name
    }
    nodes_list.append(node_record)

    # Recursively assign IDs to children
    for child in node.get('children', []):
        assign_ids(child, book_name=book_name, parent_id=node_id, page_number=page_number, nodes_list=nodes_list)

    return nodes_list

# Assign IDs and get list of nodes
nodes_with_ids = assign_ids(page_tree, page_number=1)

# Convert to DataFrame
df = pd.DataFrame(nodes_with_ids)

# Display the DataFrame
df

,node_id,parent_id,type,heading,content,page,node_type_in_tree,book_name
0,4bf6c6bb-515f-4197-9e40-ccf3d710207f,NaN,root,NaN,,1,parent,DSM5
1,bf59fb5b-0cf2-4fdc-832a-da385be50f74,4bf6c6bb-515f-4197-9e40-ccf3d710207f,paragraph,NaN,"OK, so these pointers aren’t exactly iron-clad...",1,leaf,DSM5
2,83a95966-7ffc-41df-a34f-5c01bf93a230,4bf6c6bb-515f-4197-9e40-ccf3d710207f,heading,Essential Features of Psychotic Disorder Due t...,Essential Features of Psychotic Disorder Due t...,1,parent,DSM5
3,f2d511a1-ea3c-4161-90f6-25369216de50,83a95966-7ffc-41df-a34f-5c01bf93a230,heading,Medical Condition,Medical Condition,1,parent,DSM5
4,a565d3cf-e78a-42d4-a218-70497119e051,f2d511a1-ea3c-4161-90f6-25369216de50,paragraph,NaN,"Through physiological means, a medical conditi...",1,leaf,DSM5
5,6137bf70-8590-48e3-aec0-e859032ca7b4,f2d511a1-ea3c-4161-90f6-25369216de50,heading,The Fine Print,The Fine Print,1,parent,DSM5
6,cc52b816-94b3-4fbd-80de-2010d351f9d4,6137bf70-8590-48e3-aec0-e859032ca7b4,paragraph,NaN,For pointers on deciding when a physical condi...,1,leaf,DSM5
7,4e620202-ad65-4a61-a97a-27cde8a20863,f2d511a1-ea3c-4161-90f6-25369216de50,paragraph,NaN,The D’s: • Distress or disability (work/academ...,1,leaf,DSM5
8,9eaf66b4-5ed5-4b15-9ab5-6db8c9fff2dc,f2d511a1-ea3c-4161-90f6-25369216de50,heading,Coding Notes,Coding Notes,1,parent,DSM5
9,64655367-afd1-46c0-b4e9-e5981d70747a,9eaf66b4-5ed5-4b15-9ab5-6db8c9fff2dc,paragraph,NaN,"In recording the diagnosis, use the name of th...",1,leaf,DSM5


In [80]:
nodes_with_ids = []
for i, j in missings.iterrows():
    nodes_with_ids = assign_ids(j['tree'], page_number=j['page_number'], nodes_list=nodes_with_ids,book_name=j['book_name'])
    df = pd.DataFrame(nodes_with_ids)
df

,node_id,parent_id,type,heading,content,page,node_type_in_tree,book_name
0,9be36887-a719-4869-9487-fbb517bf6079,NaN,root,NaN,,151,parent,DSM5_TR
1,ca17614b-b965-49dc-b391-38badb62b886,9be36887-a719-4869-9487-fbb517bf6079,heading,Sensory deficits.,Sensory deficits.,151,parent,DSM5_TR
2,d9f56e7c-d243-4d42-a8ba-c4427d0de208,ca17614b-b965-49dc-b391-38badb62b886,paragraph,NaN,Dysfluencies of speech may be associated with ...,151,leaf,DSM5_TR
3,2fa7dd37-5da6-450d-a5c2-2ef9b29cbe72,9be36887-a719-4869-9487-fbb517bf6079,heading,Normal speech dysfluencies.,Normal speech dysfluencies.,151,parent,DSM5_TR
4,b9de616e-5b37-408e-98ca-cd61abbb4028,2fa7dd37-5da6-450d-a5c2-2ef9b29cbe72,paragraph,NaN,The disorder must be distinguished from normal...,151,leaf,DSM5_TR
...,...,...,...,...,...,...,...,...
92,466f6751-7074-4917-bcbd-383be49e4dc8,ca0549e3-b27f-4157-b22b-481b60a734c4,heading,F17.209 Unspecified Tobacco-Related Disorder,F17.209 Unspecified Tobacco-Related Disorder,427,leaf,DSM5_ME
93,42bc762f-1783-4c53-87af-15aedd0ef140,ca0549e3-b27f-4157-b22b-481b60a734c4,heading,Other (or Unknown) Substance-Related Disorders,Other (or Unknown) Substance-Related Disorders,427,parent,DSM5_ME
94,45e6309f-9736-46f4-9a2e-79c21eac319b,42bc762f-1783-4c53-87af-15aedd0ef140,paragraph,NaN,The category of other (or unknown) substance-r...,427,leaf,DSM5_ME
95,24f93c36-3649-416a-adae-6681d3d3347c,42bc762f-1783-4c53-87af-15aedd0ef140,paragraph,NaN,Here are some examples of the substances that ...,427,parent,DSM5_ME


In [159]:
df

,node_id,parent_id,type,heading,content,page,node_type_in_tree,book_name,section,_path
0,e5cf362b-2ff0-4c7e-be69-05bd426e1938,NaN,root,Chapter 1,,34,parent,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders
1,30b04a24-ef1e-4a70-b522-9f144b51c721,e5cf362b-2ff0-4c7e-be69-05bd426e1938,heading,Neurodevelopmental Disorders,Neurodevelopmental Disorders,34,parent,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders
2,88eb8056-30b0-4204-9ec4-75b0bd74e170,30b04a24-ef1e-4a70-b522-9f144b51c721,paragraph,NaN,"Prior to DSM-5, the name of this chapter was e...",34,leaf,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders -> Neu...
3,144dd110-20d1-49e5-b628-1d223e049be7,30b04a24-ef1e-4a70-b522-9f144b51c721,heading,Quick Guide to the Neurodevelopmental Disorders,Quick Guide to the Neurodevelopmental Disorders,34,parent,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders -> Neu...
4,31e10ee3-d2a9-4ab1-ae80-9589f1e33b7c,144dd110-20d1-49e5-b628-1d223e049be7,paragraph,NaN,"In every Quick Guide, the page number followin...",34,leaf,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders -> Neu...
...,...,...,...,...,...,...,...,...,...,...
17667,466f6751-7074-4917-bcbd-383be49e4dc8,ca0549e3-b27f-4157-b22b-481b60a734c4,heading,F17.209 Unspecified Tobacco-Related Disorder,F17.209 Unspecified Tobacco-Related Disorder,427,leaf,DSM5_ME,Chapter 15: Substance-Related and Addictive Di...,Chapter 15: Substance-Related and Addictive Di...
17668,42bc762f-1783-4c53-87af-15aedd0ef140,ca0549e3-b27f-4157-b22b-481b60a734c4,heading,Other (or Unknown) Substance-Related Disorders,Other (or Unknown) Substance-Related Disorders,427,parent,DSM5_ME,Chapter 15: Substance-Related and Addictive Di...,Chapter 15: Substance-Related and Addictive Di...
17669,45e6309f-9736-46f4-9a2e-79c21eac319b,42bc762f-1783-4c53-87af-15aedd0ef140,paragraph,NaN,The category of other (or unknown) substance-r...,427,leaf,DSM5_ME,Chapter 15: Substance-Related and Addictive Di...,Chapter 15: Substance-Related and Addictive Di...
17670,24f93c36-3649-416a-adae-6681d3d3347c,42bc762f-1783-4c53-87af-15aedd0ef140,paragraph,NaN,Here are some examples of the substances that ...,427,parent,DSM5_ME,Chapter 15: Substance-Related and Addictive Di...,Chapter 15: Substance-Related and Addictive Di...


In [22]:
nodes_with_ids = []
for i, j in dsm_df.iterrows():
    nodes_with_ids = assign_ids(j['tree'], page_number=j['page_number'], nodes_list=nodes_with_ids,book_name=j['book_name'])
    df = pd.DataFrame(nodes_with_ids)
df

,node_id,parent_id,type,heading,content,page,node_type_in_tree,book_name
0,f7f2bb1f-b8f0-4796-86fb-df24f5e970fc,NaN,root,NaN,,26,parent,DSM5_ME
1,43099aac-71c3-4b15-bb63-ffab4f284ec8,f7f2bb1f-b8f0-4796-86fb-df24f5e970fc,heading,An Uncertain Diagnosis,An Uncertain Diagnosis,26,parent,DSM5_ME
2,a8969d46-3b6b-4792-b551-e6262dacb375,43099aac-71c3-4b15-bb63-ffab4f284ec8,paragraph,NaN,When you’re not sure whether a diagnosis is co...,26,leaf,DSM5_ME
3,da108059-fd58-46ba-be3c-e1bacc129196,43099aac-71c3-4b15-bb63-ffab4f284ec8,paragraph,NaN,What about a patient who comes very close to m...,26,leaf,DSM5_ME
4,456b272a-fff1-4b82-9593-1432a6fb4f9b,f7f2bb1f-b8f0-4796-86fb-df24f5e970fc,heading,Indicating Severity of a Disorder,Indicating Severity of a Disorder,26,leaf,DSM5_ME
...,...,...,...,...,...,...,...,...
25285,ed53efa4-3298-47b5-be50-d4a3ad437040,8a8bce5a-8b85-4cb4-873b-c7b90dbbfdcd,list_item,NaN,"coding of, 822",1371,leaf,DSM5_TR
25286,3a6997fa-7b7f-405b-9da5-0844f1342dc6,8a8bce5a-8b85-4cb4-873b-c7b90dbbfdcd,list_item,NaN,"contextual information in DSM-5-TR, 27",1371,leaf,DSM5_TR
25287,bb474643-6046-4f09-914e-03c066cb2032,8a8bce5a-8b85-4cb4-873b-c7b90dbbfdcd,list_item,NaN,"depressive disorders and, 190 , 195 , 199 , 20...",1371,leaf,DSM5_TR
25288,a3d3f582-c223-4ee7-bd9b-733f83c818a8,8a8bce5a-8b85-4cb4-873b-c7b90dbbfdcd,list_item,NaN,depressive episodes with short-duration hypoma...,1371,leaf,DSM5_TR


In [81]:
# Define chapter start pages based on the TOC you provided
chapter_starts_ME = [
    (0, "Also Available"),
    (3, "Title Page"),
    (4, "Copyright"),
    (5, "Dedication"),
    (6, "About the Author"),
    (7, "Acknowledgments"),
    (9, "Contents"),
    (12, "Frequently Needed Tables"),
    (13, "Introduction"),
    (34, "Chapter 1: Neurodevelopmental Disorders"),
    (64, "Chapter 2: Schizophrenia Spectrum and Other Psychotic Disorders"),  # adjust as per actual TOC
    (117, "Chapter 3: Mood Disorders"),
    (164, "Chapter 4: Anxiety Disorders"),
    (190, "Chapter 5: Obsessive–Compulsive and Related Disorders"),
    (207, "Chapter 6: Trauma- and Stressor-Related Disorders"),
    (228, "Chapter 7: Dissociative Disorders"),
    (241, "Chapter 8: Somatic Symptom and Related Disorders"),
    (267, "Chapter 9: Feeding and Eating Disorders"),
    (283, "Chapter 10: Elimination Disorders"),
    (335, "Chapter 12: Sexual Dysfunctions"),
    (356, "Chapter 13: Gender Dysphoria"),
    (363, "Chapter 14: Disruptive, Impulse-Control, and Conduct Disorders"),
    (377, "Chapter 15: Substance-Related and Addictive Disorders"),
    (436, "Chapter 16: Cognitive Disorders"),
    (490, "Chapter 17: Personality Disorders"),
    (524, "Chapter 18: Paraphilic Disorders"),
    (548, "Chapter 19: Other Factors That May Need Clinical Attention"),
    (563, "Chapter 20: Patients and Diagnoses"),
    (600, "Appendix: Essential Tables"),
    (608, "Index"),
    (631, "About Guilford Press"),
    (632, "Discover Related Guilford Books"),
]

# Sort chapters by page (just in case)
chapter_starts_ME = sorted(chapter_starts_ME, key=lambda x: x[0])

# Function to map page to chapter
def get_chapter_for_page_for_dsm5_DE(page_number):
    chapter_name = None
    for i, (start_page, name) in enumerate(chapter_starts_ME):
        # If page is before the next chapter start or last chapter
        if i + 1 < len(chapter_starts_ME):
            next_start, _ = chapter_starts_ME[i + 1]
            if start_page <= page_number < next_start:
                chapter_name = name
                break
        else:  # Last chapter
            if page_number >= start_page:
                chapter_name = name
                break
    return chapter_name

# Example usage
pages_to_test = [10, 35, 120, 365, 500, 605]
for page in pages_to_test:
    print(f"Page {page} → {get_chapter_for_page_for_dsm5_DE(page)}")

Page 10 → Contents
Page 35 → Chapter 1: Neurodevelopmental Disorders
Page 120 → Chapter 3: Mood Disorders
Page 365 → Chapter 14: Disruptive, Impulse-Control, and Conduct Disorders
Page 500 → Chapter 17: Personality Disorders
Page 605 → Appendix: Essential Tables


In [83]:
# Updated chapter start pages based on your new TOC
chapter_starts = [
    (0, "Cover Page"),
    (6, "Title Page"),
    (7, "Copyright Page"),
    (8, "Contents"),
    (10, "DSM-5-TR Chairs and Review Groups"),
    (24, "DSM-5 Task Force and Work Groups"),
    (35, "Preface to DSM-5-TR"),
    (37, "Preface to DSM-5"),
    (41, "DSM-5-TR Classification"),
    (92, "Section I DSM-5 Basics"),
    (94, "Introduction"),
    (113, "Use of the Manual"),
    (123, "Cautionary Statement for Forensic Use of DSM-5"),
    (126, "Section II Diagnostic Criteria and Codes"),
    (130, "Neurodevelopmental Disorders"),  # you suggested 130 for unclear
    (207, "Schizophrenia Spectrum and Other Psychotic Disorders"),
    (254, "Bipolar and Related Disorders"),
    (301, "Depressive Disorders"),
    (349, "Anxiety Disorders"),
    (407, "Obsessive-Compulsive and Related Disorders"),
    (447, "Trauma- and Stressor-Related Disorders"),
    (490, "Dissociative Disorders"),
    (515, "Somatic Symptom and Related Disorders"),
    (543, "Feeding and Eating Disorders"),
    (576, "Elimination Disorders"),
    (586, "Sleep-Wake Disorders"),
    (671, "Sexual Dysfunctions"),
    (712, "Gender Dysphoria"),
    (726, "Disruptive, Impulse-Control, and Conduct Disorders"),
    (753, "Substance-Related and Addictive Disorders"),
    (903, "Neurocognitive Disorders"),
    (979, "Personality Disorders"),
    (1035, "Paraphilic Disorders"),
    (1064, "Other Mental Disorders and Additional Codes"),
    (1068, "Medication-Induced Movement Disorders and Other Adverse Effects"),
    (1100, "Other Conditions That May Be a Focus of Clinical Attention"),
    (1104, "Section III Emerging Measures and Models"),
    (1107, "Assessment Measures"),
    (1124, "Culture and Psychiatric Diagnosis"),
    (1145, "Alternative DSM-5 Model for Personality Disorders"),
    (1167, "Conditions for Further Study"),
    (1197, "Appendix"),
    (1307, "Index")
]

# Sort just in case
chapter_starts = sorted(chapter_starts, key=lambda x: x[0])

# Function to map page to chapter
def get_chapter_for_page_for_dsm5_TR(page_number):
    chapter_name = "Unclear Section"  # default if no match
    for i, (start_page, name) in enumerate(chapter_starts):
        if i + 1 < len(chapter_starts):
            next_start, _ = chapter_starts[i + 1]
            if start_page <= page_number < next_start:
                chapter_name = name
                break
        else:  # Last chapter
            if page_number >= start_page:
                chapter_name = name
                break
    return chapter_name

# Example usage
pages_to_test = [5, 40, 150, 365, 580, 720, 1000]
for page in pages_to_test:
    print(f"Page {page} → {get_chapter_for_page_for_dsm5_TR(page)}")

Page 5 → Cover Page
Page 40 → Preface to DSM-5
Page 150 → Neurodevelopmental Disorders
Page 365 → Anxiety Disorders
Page 580 → Elimination Disorders
Page 720 → Gender Dysphoria
Page 1000 → Personality Disorders


In [84]:
dsm_me = ["Chapter 1: Neurodevelopmental Disorders",
"Chapter 2: Schizophrenia Spectrum and Other Psychotic Disorders",  # adjust as per actual TOC
 "Chapter 3: Mood Disorders",
 "Chapter 4: Anxiety Disorders",
 "Chapter 5: Obsessive–Compulsive and Related Disorders",
 "Chapter 6: Trauma- and Stressor-Related Disorders",
 "Chapter 7: Dissociative Disorders",
 "Chapter 8: Somatic Symptom and Related Disorders",
 "Chapter 9: Feeding and Eating Disorders",
 "Chapter 10: Elimination Disorders",
 "Chapter 12: Sexual Dysfunctions",
 "Chapter 13: Gender Dysphoria",
 "Chapter 14: Disruptive, Impulse-Control, and Conduct Disorders",
 "Chapter 15: Substance-Related and Addictive Disorders",
 "Chapter 16: Cognitive Disorders",
 "Chapter 17: Personality Disorders",
 "Chapter 18: Paraphilic Disorders",
 "Chapter 19: Other Factors That May Need Clinical Attention",
 "Chapter 20: Patients and Diagnoses"]
dsm_me

['Chapter 1: Neurodevelopmental Disorders',
 'Chapter 2: Schizophrenia Spectrum and Other Psychotic Disorders',
 'Chapter 3: Mood Disorders',
 'Chapter 4: Anxiety Disorders',
 'Chapter 5: Obsessive–Compulsive and Related Disorders',
 'Chapter 6: Trauma- and Stressor-Related Disorders',
 'Chapter 7: Dissociative Disorders',
 'Chapter 8: Somatic Symptom and Related Disorders',
 'Chapter 9: Feeding and Eating Disorders',
 'Chapter 10: Elimination Disorders',
 'Chapter 12: Sexual Dysfunctions',
 'Chapter 13: Gender Dysphoria',
 'Chapter 14: Disruptive, Impulse-Control, and Conduct Disorders',
 'Chapter 15: Substance-Related and Addictive Disorders',
 'Chapter 16: Cognitive Disorders',
 'Chapter 17: Personality Disorders',
 'Chapter 18: Paraphilic Disorders',
 'Chapter 19: Other Factors That May Need Clinical Attention',
 'Chapter 20: Patients and Diagnoses']

In [85]:
dsm5_tr_chapters = [     
    "Neurodevelopmental Disorders",  # you suggested 130 for unclear
     "Schizophrenia Spectrum and Other Psychotic Disorders",
     "Bipolar and Related Disorders",
     "Depressive Disorders",
     "Anxiety Disorders",
     "Obsessive-Compulsive and Related Disorders",
     "Trauma- and Stressor-Related Disorders",
     "Dissociative Disorders",
     "Somatic Symptom and Related Disorders",
     "Feeding and Eating Disorders",
     "Elimination Disorders",
     "Sleep-Wake Disorders",
     "Sexual Dysfunctions",
     "Gender Dysphoria",
     "Disruptive, Impulse-Control, and Conduct Disorders",
     "Substance-Related and Addictive Disorders",
     "Neurocognitive Disorders",
     "Personality Disorders",
     "Paraphilic Disorders",
     "Other Mental Disorders and Additional Codes",
     "Medication-Induced Movement Disorders and Other Adverse Effects",
     "Other Conditions That May Be a Focus of Clinical Attention",
     "Section III Emerging Measures and Models",
     "Assessment Measures",
     "Culture and Psychiatric Diagnosis",
     "Alternative DSM-5 Model for Personality Disorders",
]
dsm5_tr_chapters

['Neurodevelopmental Disorders',
 'Schizophrenia Spectrum and Other Psychotic Disorders',
 'Bipolar and Related Disorders',
 'Depressive Disorders',
 'Anxiety Disorders',
 'Obsessive-Compulsive and Related Disorders',
 'Trauma- and Stressor-Related Disorders',
 'Dissociative Disorders',
 'Somatic Symptom and Related Disorders',
 'Feeding and Eating Disorders',
 'Elimination Disorders',
 'Sleep-Wake Disorders',
 'Sexual Dysfunctions',
 'Gender Dysphoria',
 'Disruptive, Impulse-Control, and Conduct Disorders',
 'Substance-Related and Addictive Disorders',
 'Neurocognitive Disorders',
 'Personality Disorders',
 'Paraphilic Disorders',
 'Other Mental Disorders and Additional Codes',
 'Medication-Induced Movement Disorders and Other Adverse Effects',
 'Other Conditions That May Be a Focus of Clinical Attention',
 'Section III Emerging Measures and Models',
 'Assessment Measures',
 'Culture and Psychiatric Diagnosis',
 'Alternative DSM-5 Model for Personality Disorders']

In [86]:
def get_chapter_for_page(book_name, page_number):
    if(book_name=="DSM5_TR"):
        return get_chapter_for_page_for_dsm5_TR(page_number)
    else:
        return get_chapter_for_page_for_dsm5_DE(page_number)

get_chapter_for_page("DSM_DE", 1) 

'Also Available'

In [87]:
df['section'] = df.apply(lambda row: get_chapter_for_page(row['book_name'], row['page']), axis=1)
df

,node_id,parent_id,type,heading,content,page,node_type_in_tree,book_name,section
0,9be36887-a719-4869-9487-fbb517bf6079,NaN,root,NaN,,151,parent,DSM5_TR,Neurodevelopmental Disorders
1,ca17614b-b965-49dc-b391-38badb62b886,9be36887-a719-4869-9487-fbb517bf6079,heading,Sensory deficits.,Sensory deficits.,151,parent,DSM5_TR,Neurodevelopmental Disorders
2,d9f56e7c-d243-4d42-a8ba-c4427d0de208,ca17614b-b965-49dc-b391-38badb62b886,paragraph,NaN,Dysfluencies of speech may be associated with ...,151,leaf,DSM5_TR,Neurodevelopmental Disorders
3,2fa7dd37-5da6-450d-a5c2-2ef9b29cbe72,9be36887-a719-4869-9487-fbb517bf6079,heading,Normal speech dysfluencies.,Normal speech dysfluencies.,151,parent,DSM5_TR,Neurodevelopmental Disorders
4,b9de616e-5b37-408e-98ca-cd61abbb4028,2fa7dd37-5da6-450d-a5c2-2ef9b29cbe72,paragraph,NaN,The disorder must be distinguished from normal...,151,leaf,DSM5_TR,Neurodevelopmental Disorders
...,...,...,...,...,...,...,...,...,...
92,466f6751-7074-4917-bcbd-383be49e4dc8,ca0549e3-b27f-4157-b22b-481b60a734c4,heading,F17.209 Unspecified Tobacco-Related Disorder,F17.209 Unspecified Tobacco-Related Disorder,427,leaf,DSM5_ME,Chapter 15: Substance-Related and Addictive Di...
93,42bc762f-1783-4c53-87af-15aedd0ef140,ca0549e3-b27f-4157-b22b-481b60a734c4,heading,Other (or Unknown) Substance-Related Disorders,Other (or Unknown) Substance-Related Disorders,427,parent,DSM5_ME,Chapter 15: Substance-Related and Addictive Di...
94,45e6309f-9736-46f4-9a2e-79c21eac319b,42bc762f-1783-4c53-87af-15aedd0ef140,paragraph,NaN,The category of other (or unknown) substance-r...,427,leaf,DSM5_ME,Chapter 15: Substance-Related and Addictive Di...
95,24f93c36-3649-416a-adae-6681d3d3347c,42bc762f-1783-4c53-87af-15aedd0ef140,paragraph,NaN,Here are some examples of the substances that ...,427,parent,DSM5_ME,Chapter 15: Substance-Related and Addictive Di...


In [10]:
df.to_json("books/dsm-tree/tree_with_IDs.json", orient="records", lines=True)

In [14]:
import pandas as pd

df = pd.read_json("books/dsm-tree/dsm5_selected_chapters_tree.jsonl", lines=True)
df 

,node_id,parent_id,type,heading,content,page,node_type_in_tree,book_name,section,_path
0,e5cf362b-2ff0-4c7e-be69-05bd426e1938,NaN,root,Chapter 1,,34,parent,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders
1,30b04a24-ef1e-4a70-b522-9f144b51c721,e5cf362b-2ff0-4c7e-be69-05bd426e1938,heading,Neurodevelopmental Disorders,Neurodevelopmental Disorders,34,parent,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders
2,88eb8056-30b0-4204-9ec4-75b0bd74e170,30b04a24-ef1e-4a70-b522-9f144b51c721,paragraph,NaN,"Prior to DSM-5, the name of this chapter was e...",34,leaf,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders -> Neu...
3,144dd110-20d1-49e5-b628-1d223e049be7,30b04a24-ef1e-4a70-b522-9f144b51c721,heading,Quick Guide to the Neurodevelopmental Disorders,Quick Guide to the Neurodevelopmental Disorders,34,parent,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders -> Neu...
4,31e10ee3-d2a9-4ab1-ae80-9589f1e33b7c,144dd110-20d1-49e5-b628-1d223e049be7,paragraph,NaN,"In every Quick Guide, the page number followin...",34,leaf,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders -> Neu...
...,...,...,...,...,...,...,...,...,...,...
17570,0e5e1d9b-60e0-483d-be9a-5b0ece32f4ea,4b22b5f3-56d9-4b57-baf0-2fdea73c1bdd,list_item,NaN,❑ Not\n\npresent,1119,leaf,DSM5_TR,Assessment Measures,Assessment Measures -> VIII. Mania
17571,01629128-abc9-49b0-b12f-b28fd50b5624,4b22b5f3-56d9-4b57-baf0-2fdea73c1bdd,list_item,NaN,"❑ Equivocal (occasional\n\nelevated, expansive...",1119,leaf,DSM5_TR,Assessment Measures,Assessment Measures -> VIII. Mania
17572,1beb01c6-3457-440d-a121-6951586a33a0,4b22b5f3-56d9-4b57-baf0-2fdea73c1bdd,list_item,NaN,"❑ Present, but\n\nmild (frequent periods of so...",1119,leaf,DSM5_TR,Assessment Measures,Assessment Measures -> VIII. Mania
17573,d1757e9a-b3d2-40ba-b497-c74c3b71e7c9,4b22b5f3-56d9-4b57-baf0-2fdea73c1bdd,list_item,NaN,❑ Present and\n\nmoderate (frequent periods of...,1119,leaf,DSM5_TR,Assessment Measures,Assessment Measures -> VIII. Mania


In [13]:
df[(df['book_name']=='DSM5_ME') & (df['page']==35)]

,node_id,parent_id,type,heading,content,page,node_type_in_tree,book_name,section,_path


In [180]:
df.groupby(['book_name','page']).nth(-1).sort_values("page")

,node_id,parent_id,type,heading,content,page,node_type_in_tree,book_name,section,_path
10,435d3fa2-44dc-4ae4-a8a8-7abd27aae0f7,0d5aa42c-32b4-4885-9f1d-44c3e6b6c7ac,list_item,NaN,Unspecified intellectual developmental disorde...,34,leaf,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders -> Neu...
17638,d56d3510-19b9-4129-b12d-63916321797c,891dcda3-9a75-424c-a4d2-7573bae50c93,paragraph,NaN,Conduct disorder. A child persistently violate...,35,leaf,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders -> Att...
170,cfaeae33-1a2a-4349-91a2-67a6d0b4cb19,90f7c55b-f328-4b0e-bbb0-92f29d9cda9b,paragraph,NaN,**Intellectual Developmental Disorder**,36,leaf,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders -> ###...
17,33d3dfc4-de21-4972-b808-41080469ec6a,f8a3e26e-e6c9-4923-ada3-8df8759f23d4,paragraph,NaN,The many causes of IDD include genetic abnorma...,37,leaf,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders
29,5fec48f5-225f-4a20-a03a-333322a97619,232fa459-59d5-4364-889f-d55d58e89513,paragraph,NaN,"From their earliest years, people with IDD are...",38,leaf,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders -> ###...
...,...,...,...,...,...,...,...,...,...,...
17370,edeaffcd-75b0-4140-9a85-5b5a59656504,1a353e27-cddb-47e9-9813-9919addf6075,paragraph,NaN,Disorder and trait constructs each add value t...,1162,leaf,DSM5_TR,Alternative DSM-5 Model for Personality Disorders,Alternative DSM-5 Model for Personality Disord...
17466,eb863a41-5037-460c-ae2a-57410f977a2b,23b0ec71-150a-4fe9-bbaa-61ebc5dea0d2,paragraph,Intimacy,Is capable of forming and\n\ndesires to form r...,1163,leaf,DSM5_TR,Alternative DSM-5 Model for Personality Disorders,Alternative DSM-5 Model for Personality Disord...
17483,28e88104-952a-4b24-a049-052767b87e83,0530729a-f481-4857-8562-f2959df96a95,paragraph,NaN,Desire for affiliation is\n\nlimited because o...,1164,leaf,DSM5_TR,Alternative DSM-5 Model for Personality Disorders,Alternative DSM-5 Model for Personality Disord...
17507,e9299962-4d2a-4085-8c5f-454fe593d446,baac457c-65c9-403b-a7e2-b25eab471e7e,list_item,Restricted affectivity,Restricted affectivity Little reaction to emot...,1165,leaf,DSM5_TR,Alternative DSM-5 Model for Personality Disorders,Alternative DSM-5 Model for Personality Disord...


In [179]:
df.groupby(['book_name','page']).nth(1).sort_values("page")

,node_id,parent_id,type,heading,content,page,node_type_in_tree,book_name,section,_path
1,30b04a24-ef1e-4a70-b522-9f144b51c721,e5cf362b-2ff0-4c7e-be69-05bd426e1938,heading,Neurodevelopmental Disorders,Neurodevelopmental Disorders,34,parent,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders
17619,2b4cb0db-5d23-40d4-aed4-96cbe6ee2184,8ad957e1-69e6-4dd6-a465-b7c865cb0f5a,heading,Communication and Learning Disorders,Communication and Learning Disorders,35,parent,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders
153,3fe189bb-a4a6-4667-9a06-58b65b8ca518,bfa66cb7-dd66-43cd-bb6e-28c84796f258,paragraph,NaN,**Pica.** The patient eats material that is no...,36,leaf,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders
12,964440e8-d98c-4b69-8fd5-4beafab99d96,f8a3e26e-e6c9-4923-ada3-8df8759f23d4,paragraph,NaN,Individuals with intellectual developmental di...,37,leaf,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders
19,15fa21e9-f554-46ac-abce-d1e4bbe889f2,9bc86a08-a9b4-4441-910a-a6a945c5f134,paragraph,NaN,"IDD may have biological or social causes, or b...",38,leaf,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders
...,...,...,...,...,...,...,...,...,...,...
17364,2c26e2b3-3e06-4f8e-83f8-d987827624fe,1a27752c-1037-4209-a430-8be9706c90d4,paragraph,NaN,not form part of the trait definition. Importa...,1162,leaf,DSM5_TR,Alternative DSM-5 Model for Personality Disorders,Alternative DSM-5 Model for Personality Disorders
17450,292b43b1-6671-4ba4-a9b4-18a162a6376b,8e218f22-4a5d-4442-9a0b-0504cd5a3eaa,paragraph,NaN,"psychiatric diagnoses. Notably, knowing the le...",1163,leaf,DSM5_TR,Alternative DSM-5 Model for Personality Disorders,Alternative DSM-5 Model for Personality Disorders
17468,c631e5b6-47b3-4b0a-864b-d40d1574343f,219d848f-24e5-45c1-a786-584f6849927c,paragraph,NaN,esteem controlled by exaggerated concern about...,1164,leaf,DSM5_TR,Alternative DSM-5 Model for Personality Disorders,Alternative DSM-5 Model for Personality Disorders
17485,ca098fe9-5f00-4f48-8690-983e90c4b654,cbed2c31-6e2f-4075-acc4-eb71f5d03451,paragraph,NaN,distortions and confusion around self-appraisa...,1165,leaf,DSM5_TR,Alternative DSM-5 Model for Personality Disorders,Alternative DSM-5 Model for Personality Disorders


In [20]:
df2[df2['node_type_in_tree']=='leaf'].groupby('type').count()

,node_id,parent_id,heading,content,page,node_type_in_tree,book_name,section,_path
type,,,,,,,,,
case_study,128,128,48,128,128,128,128,128,128
coding_note,1,1,1,1,1,1,1,1,1
criteria_block,161,161,37,161,161,161,161,161,161
heading,470,470,470,470,470,470,470,470,470
list_item,1863,1863,167,1863,1863,1863,1863,1863,1863
note,2,2,0,2,2,2,2,2,2
paragraph,9169,9169,455,9169,9169,9169,9169,9169,9169
root,16,0,0,16,16,16,16,16,16
specifier_block,3,3,0,3,3,3,3,3,3


In [88]:

# Assuming df is your DataFrame
df['_path'] = None

# Quick lookups for content and parent_id
node_content_map = df.set_index('node_id')['content'].to_dict()
node_parent_map = df.set_index('node_id')['parent_id'].to_dict()
node_section_map = df.set_index('node_id')['section'].to_dict()

def build_path_exclude_current(node_id):
    path = []
    current_id = node_parent_map.get(node_id)  # start from the parent of current node

    # Traverse up until the root
    while pd.notna(current_id):
        content = node_content_map.get(current_id)
        if content and str(content).strip() != "":
            path.insert(0, content)  # prepend to keep hierarchical order
        current_id = node_parent_map.get(current_id)

    # Prepend Section at the start
    section = node_section_map.get(node_id)
    if section:
        path.insert(0, section)

    return ' -> '.join(path)

# Apply to all rows
df['_path'] = df['node_id'].apply(build_path_exclude_current)

# Preview
prew = df[['content', 'section', '_path','node_type_in_tree']]
prew

,content,section,_path,node_type_in_tree
0,,Neurodevelopmental Disorders,Neurodevelopmental Disorders,parent
1,Sensory deficits.,Neurodevelopmental Disorders,Neurodevelopmental Disorders,parent
2,Dysfluencies of speech may be associated with ...,Neurodevelopmental Disorders,Neurodevelopmental Disorders -> Sensory deficits.,leaf
3,Normal speech dysfluencies.,Neurodevelopmental Disorders,Neurodevelopmental Disorders,parent
4,The disorder must be distinguished from normal...,Neurodevelopmental Disorders,Neurodevelopmental Disorders -> Normal speech ...,leaf
...,...,...,...,...
92,F17.209 Unspecified Tobacco-Related Disorder,Chapter 15: Substance-Related and Addictive Di...,Chapter 15: Substance-Related and Addictive Di...,leaf
93,Other (or Unknown) Substance-Related Disorders,Chapter 15: Substance-Related and Addictive Di...,Chapter 15: Substance-Related and Addictive Di...,parent
94,The category of other (or unknown) substance-r...,Chapter 15: Substance-Related and Addictive Di...,Chapter 15: Substance-Related and Addictive Di...,leaf
95,Here are some examples of the substances that ...,Chapter 15: Substance-Related and Addictive Di...,Chapter 15: Substance-Related and Addictive Di...,parent


In [90]:
df[df['node_type_in_tree']=='leaf'].loc[50]

node_id                           ac718203-696d-44f0-9968-6b9136577bd0
parent_id                         2b4cb0db-5d23-40d4-aed4-96cbe6ee2184
type                                                         paragraph
heading                                                            NaN
content              Specific learning disorder. This may involve p...
page                                                                35
node_type_in_tree                                                 leaf
book_name                                                      DSM5_ME
section                        Chapter 1: Neurodevelopmental Disorders
_path                Chapter 1: Neurodevelopmental Disorders -> Com...
Name: 50, dtype: object

In [91]:
df[df['node_type_in_tree']=='leaf']

,node_id,parent_id,type,heading,content,page,node_type_in_tree,book_name,section,_path
2,d9f56e7c-d243-4d42-a8ba-c4427d0de208,ca17614b-b965-49dc-b391-38badb62b886,paragraph,NaN,Dysfluencies of speech may be associated with ...,151,leaf,DSM5_TR,Neurodevelopmental Disorders,Neurodevelopmental Disorders -> Sensory deficits.
4,b9de616e-5b37-408e-98ca-cd61abbb4028,2fa7dd37-5da6-450d-a5c2-2ef9b29cbe72,paragraph,NaN,The disorder must be distinguished from normal...,151,leaf,DSM5_TR,Neurodevelopmental Disorders,Neurodevelopmental Disorders -> Normal speech ...
6,f547ead0-d644-4470-ac3e-f150a882baa2,daa15cfe-a091-4d0a-bdcf-1412aa5eb427,paragraph,NaN,Children who have dysfluencies when they read ...,151,leaf,DSM5_TR,Neurodevelopmental Disorders,Neurodevelopmental Disorders -> Specific learn...
8,bf7848bb-b037-45ec-a56f-ce0217f1ee89,69c2989b-3cec-4cad-8c36-3279ef88540d,paragraph,NaN,It is necessary to distinguish between dysflue...,151,leaf,DSM5_TR,Neurodevelopmental Disorders,Neurodevelopmental Disorders -> Bilingualism.
10,1fc6b67d-db19-4b12-aacd-7bfbcf1231b6,f25ddb37-4557-426c-986c-428a8158b936,paragraph,NaN,Stuttering may occur as a side effect of medic...,151,leaf,DSM5_TR,Neurodevelopmental Disorders,Neurodevelopmental Disorders -> Medication sid...
...,...,...,...,...,...,...,...,...,...,...
89,eea86cac-503b-44d4-ac71-37609d8e25c0,bfd74629-a23f-4800-9f20-8bb1576b8791,paragraph,NaN,You can find the specifics of tobacco withdraw...,427,leaf,DSM5_ME,Chapter 15: Substance-Related and Addictive Di...,Chapter 15: Substance-Related and Addictive Di...
91,ec9fae79-5787-48ba-8bf5-56128e9a006c,835bf6dc-36d7-480e-8926-e2b325759868,paragraph,NaN,The other tobacco-induced disorders are listed...,427,leaf,DSM5_ME,Chapter 15: Substance-Related and Addictive Di...,Chapter 15: Substance-Related and Addictive Di...
92,466f6751-7074-4917-bcbd-383be49e4dc8,ca0549e3-b27f-4157-b22b-481b60a734c4,heading,F17.209 Unspecified Tobacco-Related Disorder,F17.209 Unspecified Tobacco-Related Disorder,427,leaf,DSM5_ME,Chapter 15: Substance-Related and Addictive Di...,Chapter 15: Substance-Related and Addictive Di...
94,45e6309f-9736-46f4-9a2e-79c21eac319b,42bc762f-1783-4c53-87af-15aedd0ef140,paragraph,NaN,The category of other (or unknown) substance-r...,427,leaf,DSM5_ME,Chapter 15: Substance-Related and Addictive Di...,Chapter 15: Substance-Related and Addictive Di...


In [101]:
df

,node_id,parent_id,type,heading,content,page,node_type_in_tree,book_name,section,_path
0,9be36887-a719-4869-9487-fbb517bf6079,NaN,root,NaN,,151,parent,DSM5_TR,Neurodevelopmental Disorders,Neurodevelopmental Disorders
1,ca17614b-b965-49dc-b391-38badb62b886,9be36887-a719-4869-9487-fbb517bf6079,heading,Sensory deficits.,Sensory deficits.,151,parent,DSM5_TR,Neurodevelopmental Disorders,Neurodevelopmental Disorders
2,d9f56e7c-d243-4d42-a8ba-c4427d0de208,ca17614b-b965-49dc-b391-38badb62b886,paragraph,NaN,Dysfluencies of speech may be associated with ...,151,leaf,DSM5_TR,Neurodevelopmental Disorders,Neurodevelopmental Disorders -> Sensory deficits.
3,2fa7dd37-5da6-450d-a5c2-2ef9b29cbe72,9be36887-a719-4869-9487-fbb517bf6079,heading,Normal speech dysfluencies.,Normal speech dysfluencies.,151,parent,DSM5_TR,Neurodevelopmental Disorders,Neurodevelopmental Disorders
4,b9de616e-5b37-408e-98ca-cd61abbb4028,2fa7dd37-5da6-450d-a5c2-2ef9b29cbe72,paragraph,NaN,The disorder must be distinguished from normal...,151,leaf,DSM5_TR,Neurodevelopmental Disorders,Neurodevelopmental Disorders -> Normal speech ...
...,...,...,...,...,...,...,...,...,...,...
92,466f6751-7074-4917-bcbd-383be49e4dc8,ca0549e3-b27f-4157-b22b-481b60a734c4,heading,F17.209 Unspecified Tobacco-Related Disorder,F17.209 Unspecified Tobacco-Related Disorder,427,leaf,DSM5_ME,Chapter 15: Substance-Related and Addictive Di...,Chapter 15: Substance-Related and Addictive Di...
93,42bc762f-1783-4c53-87af-15aedd0ef140,ca0549e3-b27f-4157-b22b-481b60a734c4,heading,Other (or Unknown) Substance-Related Disorders,Other (or Unknown) Substance-Related Disorders,427,parent,DSM5_ME,Chapter 15: Substance-Related and Addictive Di...,Chapter 15: Substance-Related and Addictive Di...
94,45e6309f-9736-46f4-9a2e-79c21eac319b,42bc762f-1783-4c53-87af-15aedd0ef140,paragraph,NaN,The category of other (or unknown) substance-r...,427,leaf,DSM5_ME,Chapter 15: Substance-Related and Addictive Di...,Chapter 15: Substance-Related and Addictive Di...
95,24f93c36-3649-416a-adae-6681d3d3347c,42bc762f-1783-4c53-87af-15aedd0ef140,paragraph,NaN,Here are some examples of the substances that ...,427,parent,DSM5_ME,Chapter 15: Substance-Related and Addictive Di...,Chapter 15: Substance-Related and Addictive Di...


In [164]:
df_miss = pd.read_json("books/dsm-tree/missing_leaf.jsonl", lines=True)
df_miss

,node_id,parent_id,type,heading,content,page,node_type_in_tree,book_name,section,_path
0,9be36887-a719-4869-9487-fbb517bf6079,NaN,root,NaN,,151,parent,DSM5_TR,Neurodevelopmental Disorders,Neurodevelopmental Disorders
1,ca17614b-b965-49dc-b391-38badb62b886,9be36887-a719-4869-9487-fbb517bf6079,heading,Sensory deficits.,Sensory deficits.,151,parent,DSM5_TR,Neurodevelopmental Disorders,Neurodevelopmental Disorders
2,d9f56e7c-d243-4d42-a8ba-c4427d0de208,ca17614b-b965-49dc-b391-38badb62b886,paragraph,NaN,Dysfluencies of speech may be associated with ...,151,leaf,DSM5_TR,Neurodevelopmental Disorders,Neurodevelopmental Disorders -> Sensory deficits.
3,2fa7dd37-5da6-450d-a5c2-2ef9b29cbe72,9be36887-a719-4869-9487-fbb517bf6079,heading,Normal speech dysfluencies.,Normal speech dysfluencies.,151,parent,DSM5_TR,Neurodevelopmental Disorders,Neurodevelopmental Disorders
4,b9de616e-5b37-408e-98ca-cd61abbb4028,2fa7dd37-5da6-450d-a5c2-2ef9b29cbe72,paragraph,NaN,The disorder must be distinguished from normal...,151,leaf,DSM5_TR,Neurodevelopmental Disorders,Neurodevelopmental Disorders -> Normal speech ...
...,...,...,...,...,...,...,...,...,...,...
92,466f6751-7074-4917-bcbd-383be49e4dc8,ca0549e3-b27f-4157-b22b-481b60a734c4,heading,F17.209 Unspecified Tobacco-Related Disorder,F17.209 Unspecified Tobacco-Related Disorder,427,leaf,DSM5_ME,Chapter 15: Substance-Related and Addictive Di...,Chapter 15: Substance-Related and Addictive Di...
93,42bc762f-1783-4c53-87af-15aedd0ef140,ca0549e3-b27f-4157-b22b-481b60a734c4,heading,Other (or Unknown) Substance-Related Disorders,Other (or Unknown) Substance-Related Disorders,427,parent,DSM5_ME,Chapter 15: Substance-Related and Addictive Di...,Chapter 15: Substance-Related and Addictive Di...
94,45e6309f-9736-46f4-9a2e-79c21eac319b,42bc762f-1783-4c53-87af-15aedd0ef140,paragraph,NaN,The category of other (or unknown) substance-r...,427,leaf,DSM5_ME,Chapter 15: Substance-Related and Addictive Di...,Chapter 15: Substance-Related and Addictive Di...
95,24f93c36-3649-416a-adae-6681d3d3347c,42bc762f-1783-4c53-87af-15aedd0ef140,paragraph,NaN,Here are some examples of the substances that ...,427,parent,DSM5_ME,Chapter 15: Substance-Related and Addictive Di...,Chapter 15: Substance-Related and Addictive Di...


In [172]:
df = pd.read_json("books/dsm-tree/dsm5_selected_chapters_tree.jsonl", lines=True)
df

,node_id,parent_id,type,heading,content,page,node_type_in_tree,book_name,section,_path
0,e5cf362b-2ff0-4c7e-be69-05bd426e1938,NaN,root,Chapter 1,,34,parent,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders
1,30b04a24-ef1e-4a70-b522-9f144b51c721,e5cf362b-2ff0-4c7e-be69-05bd426e1938,heading,Neurodevelopmental Disorders,Neurodevelopmental Disorders,34,parent,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders
2,88eb8056-30b0-4204-9ec4-75b0bd74e170,30b04a24-ef1e-4a70-b522-9f144b51c721,paragraph,NaN,"Prior to DSM-5, the name of this chapter was e...",34,leaf,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders -> Neu...
3,144dd110-20d1-49e5-b628-1d223e049be7,30b04a24-ef1e-4a70-b522-9f144b51c721,heading,Quick Guide to the Neurodevelopmental Disorders,Quick Guide to the Neurodevelopmental Disorders,34,parent,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders -> Neu...
4,31e10ee3-d2a9-4ab1-ae80-9589f1e33b7c,144dd110-20d1-49e5-b628-1d223e049be7,paragraph,NaN,"In every Quick Guide, the page number followin...",34,leaf,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders -> Neu...
...,...,...,...,...,...,...,...,...,...,...
17667,466f6751-7074-4917-bcbd-383be49e4dc8,ca0549e3-b27f-4157-b22b-481b60a734c4,heading,F17.209 Unspecified Tobacco-Related Disorder,F17.209 Unspecified Tobacco-Related Disorder,427,leaf,DSM5_ME,Chapter 15: Substance-Related and Addictive Di...,Chapter 15: Substance-Related and Addictive Di...
17668,42bc762f-1783-4c53-87af-15aedd0ef140,ca0549e3-b27f-4157-b22b-481b60a734c4,heading,Other (or Unknown) Substance-Related Disorders,Other (or Unknown) Substance-Related Disorders,427,parent,DSM5_ME,Chapter 15: Substance-Related and Addictive Di...,Chapter 15: Substance-Related and Addictive Di...
17669,45e6309f-9736-46f4-9a2e-79c21eac319b,42bc762f-1783-4c53-87af-15aedd0ef140,paragraph,NaN,The category of other (or unknown) substance-r...,427,leaf,DSM5_ME,Chapter 15: Substance-Related and Addictive Di...,Chapter 15: Substance-Related and Addictive Di...
17670,24f93c36-3649-416a-adae-6681d3d3347c,42bc762f-1783-4c53-87af-15aedd0ef140,paragraph,NaN,Here are some examples of the substances that ...,427,parent,DSM5_ME,Chapter 15: Substance-Related and Addictive Di...,Chapter 15: Substance-Related and Addictive Di...


In [175]:
df[df['page']==427]

,node_id,parent_id,type,heading,content,page,node_type_in_tree,book_name,section,_path
8658,6da3eae0-4fc7-4b31-8c88-8ffcfe1dfeff,NaN,root,NaN,,427,parent,DSM5_TR,Obsessive-Compulsive and Related Disorders,Obsessive-Compulsive and Related Disorders
8659,19ae7074-26ef-4658-9bf2-2f575c29f33a,6da3eae0-4fc7-4b31-8c88-8ffcfe1dfeff,paragraph,NaN,the lack of clutter is attributable to a third...,427,leaf,DSM5_TR,Obsessive-Compulsive and Related Disorders,Obsessive-Compulsive and Related Disorders
8660,a9880895-8339-4b8e-84f1-fe662641fab0,6da3eae0-4fc7-4b31-8c88-8ffcfe1dfeff,paragraph,NaN,"Symptoms (i.e., difficulty discarding and/or c...",427,leaf,DSM5_TR,Obsessive-Compulsive and Related Disorders,Obsessive-Compulsive and Related Disorders
8661,e1351890-660b-475e-85d4-6ae05cd3c4b8,6da3eae0-4fc7-4b31-8c88-8ffcfe1dfeff,heading,Associated Features,Associated Features,427,parent,DSM5_TR,Obsessive-Compulsive and Related Disorders,Obsessive-Compulsive and Related Disorders
8662,632f43da-e4f5-4adf-bcbf-36a1d482c568,e1351890-660b-475e-85d4-6ae05cd3c4b8,paragraph,NaN,Other common features of hoarding disorder inc...,427,leaf,DSM5_TR,Obsessive-Compulsive and Related Disorders,Obsessive-Compulsive and Related Disorders -> ...
8663,a9271bef-8af7-4078-848a-88e1ff533d1a,6da3eae0-4fc7-4b31-8c88-8ffcfe1dfeff,heading,Prevalence,Prevalence,427,parent,DSM5_TR,Obsessive-Compulsive and Related Disorders,Obsessive-Compulsive and Related Disorders
8664,37a6dc59-1f8e-491b-a151-7d82da0f49ab,a9271bef-8af7-4078-848a-88e1ff533d1a,paragraph,NaN,Nationally representative prevalence studies o...,427,leaf,DSM5_TR,Obsessive-Compulsive and Related Disorders,Obsessive-Compulsive and Related Disorders -> ...
8665,039f58bb-e028-4a85-a243-cefd9e7d605d,6da3eae0-4fc7-4b31-8c88-8ffcfe1dfeff,heading,Development and Course,Development and Course,427,parent,DSM5_TR,Obsessive-Compulsive and Related Disorders,Obsessive-Compulsive and Related Disorders
8666,84915319-49ac-40a3-be82-981faf6a94a1,039f58bb-e028-4a85-a243-cefd9e7d605d,paragraph,NaN,Hoarding appears to begin early in life and sp...,427,leaf,DSM5_TR,Obsessive-Compulsive and Related Disorders,Obsessive-Compulsive and Related Disorders -> ...
17655,ca0549e3-b27f-4157-b22b-481b60a734c4,NaN,root,NaN,,427,parent,DSM5_ME,Chapter 15: Substance-Related and Addictive Di...,Chapter 15: Substance-Related and Addictive Di...


In [169]:
dfff = pd.concat([df_miss, df])
dfff[dfff['page']==35]

,node_id,parent_id,type,heading,content,page,node_type_in_tree,book_name,section,_path
43,8ad957e1-69e6-4dd6-a465-b7c865cb0f5a,NaN,root,NaN,,35,parent,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders
44,2b4cb0db-5d23-40d4-aed4-96cbe6ee2184,8ad957e1-69e6-4dd6-a465-b7c865cb0f5a,heading,Communication and Learning Disorders,Communication and Learning Disorders,35,parent,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders
45,7842b97b-96bb-49d4-9f5f-3de053ca91ce,2b4cb0db-5d23-40d4-aed4-96cbe6ee2184,paragraph,NaN,Language disorder. A child’s delay in using sp...,35,leaf,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders -> Com...
46,db84276a-6fc8-4778-b9cd-439be0609e61,2b4cb0db-5d23-40d4-aed4-96cbe6ee2184,paragraph,NaN,Social (pragmatic) communication disorder. Des...,35,leaf,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders -> Com...
47,ff37cc90-befe-479c-b5df-0d8650b824b7,2b4cb0db-5d23-40d4-aed4-96cbe6ee2184,paragraph,NaN,Speech sound disorder. Difficulty producing th...,35,leaf,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders -> Com...
48,0b16ea9f-6317-456e-b00f-33dd09cb9154,2b4cb0db-5d23-40d4-aed4-96cbe6ee2184,paragraph,NaN,Childhood-onset fluency disorder (stuttering)....,35,leaf,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders -> Com...
49,be34a068-0e5b-4171-aa11-7827e0ca3457,2b4cb0db-5d23-40d4-aed4-96cbe6ee2184,paragraph,NaN,"Selective mutism. A child chooses not to talk,...",35,leaf,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders -> Com...
50,ac718203-696d-44f0-9968-6b9136577bd0,2b4cb0db-5d23-40d4-aed4-96cbe6ee2184,paragraph,NaN,Specific learning disorder. This may involve p...,35,leaf,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders -> Com...
51,b497e85b-1167-4661-9dbf-63827de5aed4,2b4cb0db-5d23-40d4-aed4-96cbe6ee2184,paragraph,NaN,Academic or educational problems. These Z-code...,35,leaf,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders -> Com...
52,0ca6ff5c-b25c-4c25-aacd-82ad1b3ce843,2b4cb0db-5d23-40d4-aed4-96cbe6ee2184,paragraph,NaN,Unspecified communication disorder. Use for co...,35,leaf,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders -> Com...


In [100]:
df.to_json("books/dsm-tree/missing_leaf.jsonl", orient='records', lines=True)

In [76]:
df[df['book_name']=='DSM5_ME'].sort_values("page")

,node_id,parent_id,type,heading,content,page,node_type_in_tree,book_name,section,_path
11,ede348ff-d64b-44a7-9099-1e2c4ea9ea2f,8e85080d-9705-4c73-8b2c-4729e836952a,paragraph,NaN,When Psychological Problems Mask Medical Disor...,2,leaf,DSM5_ME,Also Available,Also Available -> Also from James Morrison
12,c022f234-1c81-425a-b7e8-0350b9eb3c3b,8e85080d-9705-4c73-8b2c-4729e836952a,paragraph,NaN,"For more information, see *www.guilford.com/mo...",2,leaf,DSM5_ME,Also Available,Also Available -> Also from James Morrison
6,8e85080d-9705-4c73-8b2c-4729e836952a,350ffe58-9c7b-4783-867b-61ecbf5df98d,heading,Also from James Morrison,Also from James Morrison,2,parent,DSM5_ME,Also Available,Also Available
7,e5d89330-82ef-415a-bac6-12042ccac85f,8e85080d-9705-4c73-8b2c-4729e836952a,paragraph,NaN,Diagnosis Made Easier:\n\nPrinciples and Techn...,2,leaf,DSM5_ME,Also Available,Also Available -> Also from James Morrison
9,77ba06cc-1ef7-4cbd-b4a6-12d05979cbfb,8e85080d-9705-4c73-8b2c-4729e836952a,paragraph,NaN,Interviewing Children and Adolescents:\n\nSkil...,2,leaf,DSM5_ME,Also Available,Also Available -> Also from James Morrison
...,...,...,...,...,...,...,...,...,...,...
6306,a9ca856e-2c7d-4397-a830-04ebbfc5752d,07203637-4bd4-4b62-ba74-3d0b2b56ddd7,heading,Guilford Journals Online,Guilford Journals Online,632,parent,DSM5_ME,Discover Related Guilford Books,Discover Related Guilford Books
6186,75a1ff6e-839f-4c29-8e80-1692fbcd2dee,a8e5b984-afb5-476d-bec1-fb549751540a,paragraph,NaN,The diagram shows a rectangular bar to denote ...,633,leaf,DSM5_ME,Discover Related Guilford Books,Discover Related Guilford Books
6187,a2d11484-ef1a-4d4c-9f35-f0c0a92c9f67,a8e5b984-afb5-476d-bec1-fb549751540a,paragraph,NaN,Psychosis symptoms are denoted by backward sla...,633,leaf,DSM5_ME,Discover Related Guilford Books,Discover Related Guilford Books
6185,9caa8f80-c8cb-4404-8f4a-6709c44f1907,a8e5b984-afb5-476d-bec1-fb549751540a,paragraph,NaN,Extended image description for Page 89,633,leaf,DSM5_ME,Discover Related Guilford Books,Discover Related Guilford Books


In [77]:
_samp[_samp['node_type_in_tree']=='parent']['content'].tolist(), _samp[_samp['node_type_in_tree']=='parent']['heading'].tolist()

(['',
  'Essential Features of **Schizoaffective Disorder**',
  '**The Fine Print**',
  '**Coding Notes**'],
 [nan,
  'Essential Features of Schizoaffective Disorder',
  'The Fine Print',
  'Coding Notes'])

In [80]:
df[df["page"]==134]

,node_id,parent_id,type,heading,content,page,node_type_in_tree,book_name,section,_path
1019,01aa56a7-7d50-4314-90f3-11e21ecc92d3,NaN,root,NaN,,134,parent,DSM5_ME,Chapter 3: Mood Disorders,Chapter 3: Mood Disorders
1020,1fb434f3-304e-412d-b0e4-b3ca202409ae,01aa56a7-7d50-4314-90f3-11e21ecc92d3,paragraph,NaN,"out for any postpartum patient, but she seems ...",134,leaf,DSM5_ME,Chapter 3: Mood Disorders,Chapter 3: Mood Disorders
1021,60bc383e-67bb-420d-b539-ee6be3f1428b,01aa56a7-7d50-4314-90f3-11e21ecc92d3,case_study,Winona Fisk,"By the time she turned 21, Winona Fisk had alr...",134,leaf,DSM5_ME,Chapter 3: Mood Disorders,Chapter 3: Mood Disorders
1022,3111bfa4-6685-4237-93d7-30eb90103c68,01aa56a7-7d50-4314-90f3-11e21ecc92d3,heading,Evaluation of Winona Fisk,Evaluation of Winona Fisk,134,parent,DSM5_ME,Chapter 3: Mood Disorders,Chapter 3: Mood Disorders
1023,68f4ca53-0a0e-43a3-8ee6-8a18d56025f3,3111bfa4-6685-4237-93d7-30eb90103c68,paragraph,NaN,Winona’s two previous episodes of bipolar I di...,134,leaf,DSM5_ME,Chapter 3: Mood Disorders,Chapter 3: Mood Disorders -> Evaluation of Win...
8732,9193d205-5df5-4958-b79c-6289f9db4156,NaN,root,NaN,,134,parent,DSM5_TR,Neurodevelopmental Disorders,Neurodevelopmental Disorders
8733,3fcdcb9c-31ff-474c-9e57-941b26c77263,9193d205-5df5-4958-b79c-6289f9db4156,heading,**Diagnostic Criteria**,**Diagnostic Criteria**,134,parent,DSM5_TR,Neurodevelopmental Disorders,Neurodevelopmental Disorders
8734,f2517e10-748a-49d0-a760-4a28e6e5ae59,3fcdcb9c-31ff-474c-9e57-941b26c77263,paragraph,NaN,Intellectual developmental disorder (intellect...,134,leaf,DSM5_TR,Neurodevelopmental Disorders,Neurodevelopmental Disorders -> **Diagnostic C...
8735,18f0d3ce-3071-4da6-af9c-c6a6fdc43005,3fcdcb9c-31ff-474c-9e57-941b26c77263,criteria_block,NaN,,134,parent,DSM5_TR,Neurodevelopmental Disorders,Neurodevelopmental Disorders -> **Diagnostic C...
8736,79cc4706-14e0-4a83-ad63-00beba9e7740,18f0d3ce-3071-4da6-af9c-c6a6fdc43005,list_item,NaN,"A. Deficits in intellectual functions, such as...",134,leaf,DSM5_TR,Neurodevelopmental Disorders,Neurodevelopmental Disorders -> **Diagnostic C...


In [ ]:
df.groupby("")

In [ ]:


lm = dspy.LM(
    model=settings.llm_model,
    api_base=settings.llm_api_base,
    api_key=settings.llm_api_key,
    timeout=settings.llm_timeout,
)
dspy.settings.configure(lm=lm, temperature=0.0)


In [22]:
from signatures.entity_extraction import DisorderEntityExtractor
from signatures.entity_example import case_study_para2_entities, case_study_para1_entities, example_entities
from dspy.teleprompt import BootstrapFewShot

trainset = [case_study_para2_entities, case_study_para1_entities, example_entities]
cot = dspy.ChainOfThought(DisorderEntityExtractor)
optimizer = BootstrapFewShot()
compiled_cot = optimizer.compile(
    cot,
    trainset=trainset
)

  0%|          | 0/3 [00:00<?, ?it/s]

2026/05/13 17:10:18 ERROR dspy.teleprompt.bootstrap: Failed to run or to evaluate example Example({'text': 'When Julian was a young teenager, his dad died. "His death was self-inflicted," Julian points out. "He\'d had rheumatic fever as a child, which gave him an enlarged heart. And the only thing he ever exercised was his right to eat anything fried, including Twinkies. And he smoked—he was a proud two-pack-a-day man. Look where that got him." None of these health risks apply to Julian, who is nothing if not careful about what he puts into his body. He has spent hours searching the Internet for information on diet, and he once attended a lecture by Dean Ornish. "I\'ve followed a plant-based diet ever since," Julian said. "I\'m especially keen on tofu. And broccoli." Julian has never complained much about having symptoms—just the odd palpitation, maybe "hot flushes" on an especially humid day. "I don\'t feel bad," he explains. "I just feel scared." This time, he\'s heard a report on NP

Bootstrapped 0 full traces after 2 examples for up to 1 rounds, amounting to 3 attempts.


In [113]:
import pandas as pd


df = pd.read_json("./books/dsm5-KG/dsm5-KG.jsonl", lines=True)
df

,0_entity,1_entity,2_entity,3_entity,4_entity,5_entity,6_entity,7_entity,8_entity,node_id,...,33_relation,34_relation,35_relation,36_relation,37_relation,38_relation,39_relation,40_relation,41_relation,42_relation
0,"{'name': 'Body dysmorphic disorder', 'type': '...","{'name': 'appearance-related preoccupations', ...","{'name': 'repetitive behaviors', 'type': 'symp...","{'name': 'clinically significant distress', 't...","{'name': 'impairment in functioning', 'type': ...","{'name': 'Physical defects', 'type': 'concept'}","{'name': 'skin picking', 'type': 'symptom'}","{'name': 'skin lesions', 'type': 'symptom'}","{'name': 'scarring', 'type': 'symptom'}",7cba5858-4550-43d3-9ffa-eb8b59cc2eed,...,None,None,None,None,None,None,None,None,None,None
1,"{'name': 'committing suicide', 'type': 'symptom'}","{'name': 'B', 'type': 'criterion'}","{'name': 'Mixed symptoms', 'type': 'symptom'}","{'name': 'C', 'type': 'criterion'}","{'name': 'mania', 'type': 'disorder'}","{'name': 'depression', 'type': 'disorder'}","{'name': 'manic episode, with mixed features',...","{'name': 'marked impairment', 'type': 'criteri...","{'name': 'clinical severity', 'type': 'criteri...",0ec02fd1-7cc3-49d1-839e-714ce38e563c,...,None,None,None,None,None,None,None,None,None,None
2,"{'name': 'Russell', 'type': 'patient'}","{'name': 'sweaty', 'type': 'symptom'}","{'name': 'chills', 'type': 'symptom'}","{'name': 'headache', 'type': 'symptom'}","{'name': 'cramping abdominal pain', 'type': 's...","{'name': 'flu', 'type': 'disorder'}","{'name': 'agitated complaint', 'type': 'symptom'}","{'name': 'no fever', 'type': 'symptom'}","{'name': 'pothead coming unglued', 'type': 'co...",4c279985-76de-41a4-88fa-6500fca649d0,...,None,None,None,None,None,None,None,None,None,None
3,"{'name': 'male', 'type': 'concept'}","{'name': 'childhood', 'type': 'duration'}","{'name': 'adolescence', 'type': 'duration'}",{'name': 'interest in various aspects of fire'...,"{'name': 'false alarms', 'type': 'symptom'}","{'name': 'spectators at fires', 'type': 'sympt...",{'name': 'collect the apparatus used by firefi...,"{'name': 'volunteer firefighters', 'type': 'co...","{'name': 'Pyromania', 'type': 'disorder'}",a2d9bb4f-9efc-4914-bb78-433dec1d2d5d,...,None,None,None,None,None,None,None,None,None,None
4,"{'name': 'phencyclidine intoxication', 'type':...",{'name': 'phencyclidine-induced mental disorde...,"{'name': 'heavy use', 'type': 'concept'}","{'name': 'phencyclidine use disorder', 'type':...",None,None,None,None,None,22b0bd97-0d32-4af9-932f-ba69b17cce4c,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10968,"{'name': 'Cannabis', 'type': 'concept'}",{'name': 'taken in larger amounts or over a lo...,None,None,None,None,None,None,None,f544dbb8-3496-4dd1-8b18-a6e3a3e85839,...,None,None,None,None,None,None,None,None,None,None
10969,"{'name': 'Severity', 'type': 'specifier'}","{'name': 'psychosis', 'type': 'concept'}","{'name': 'delusions', 'type': 'symptom'}","{'name': 'hallucinations', 'type': 'symptom'}","{'name': 'disorganized speech', 'type': 'sympt...","{'name': 'abnormal psychomotor behavior', 'typ...","{'name': 'negative symptoms', 'type': 'symptom'}","{'name': 'last 7 days', 'type': 'duration'}","{'name': 'delusional disorder', 'type': 'disor...",72c5a9e9-0aaa-4ca2-ad83-4e63c7821436,...,None,None,None,None,None,None,None,None,None,None
10970,{'name': 'substance/medication-induced bipolar...,"{'name': 'bipolar I disorder', 'type': 'disord...","{'name': 'substance', 'type': 'concept'}","{'name': 'stimulants', 'type': 'concept'}","{'name': 'phencyclidine', 'type': 'concept'}","{'name': 'medication', 'type': 'concept'}","{'name': 'steroids', 'type': 'concept'}","{'name': 'manic episode', 'type': 'symptom'}","{'name': 'manic symptoms', 'type': 'symptom'}",b71ddd68-e207-43d1-bef0-74249f3aebdd,...,None,None,None,None,None,None,None,None,None,None
10971,"{'name': 'Tics', 'type': 'symptom'}","{'name': 'twitch', '

In [110]:
df

In [114]:
df2 = pd.read_json("books/dsm5-KG/missings_tree.jsonl", lines=True)
df2

,0_entity,1_entity,2_entity,3_entity,4_entity,5_entity,node_id,6_entity,7_entity,8_entity,...,2_relation,3_relation,4_relation,5_relation,6_relation,7_relation,8_relation,9_relation,10_relation,11_relation
0,"{'name': 'Illness anxiety disorder', 'type': '...","{'name': 'hypochondriasis', 'type': 'disorder'}","{'name': 'unfounded fear of a serious, often l...","{'name': 'cancer', 'type': 'disorder'}","{'name': 'heart disease', 'type': 'disorder'}","{'name': 'somatic symptoms', 'type': 'symptom'}",1c42e031-05d0-482c-bd07-54e4cbaa7dbe,None,None,None,...,"{'subject': 'unfounded fear of a serious, ofte...","{'subject': 'unfounded fear of a serious, ofte...","{'subject': 'Illness anxiety disorder', 'predi...",None,None,None,None,None,None,None
1,"{'name': 'Vocal tics', 'type': 'symptom'}","{'name': 'repetitive vocalizations', 'type': '...","{'name': 'Tourette’s disorder', 'type': 'disor...","{'name': 'repetitive sounds', 'type': 'symptom'}","{'name': 'childhood-onset fluency disorder', '...",None,f4b7f9dd-2f0d-4c5a-a249-2dbd45c3418d,None,None,None,...,{'subject': 'childhood-onset fluency disorder'...,"{'subject': 'Tourette’s disorder', 'predicate'...","{'subject': 'Vocal tics', 'predicate': 'distin...","{'subject': 'repetitive vocalizations', 'predi...",None,None,None,None,None,None
2,"{'name': 'Actual physical illness', 'type': 'c...","{'name': 'Psychological causes', 'type': 'conc...","{'name': 'physical symptoms', 'type': 'symptom'}","{'name': 'physical disorders', 'type': 'disord...",None,None,15d97c53-1d5d-49e5-abad-d4ee6b910fd9,None,None,None,...,None,None,None,None,None,None,None,None,None,None
3,"{'name': '4', 'type': 'criterion'}","{'name': 'Exaggerated startle response', 'type...",None,None,None,None,a8a1244a-801a-4e19-b5d3-edacaf5b30aa,None,None,None,...,None,None,None,None,None,None,None,None,None,None
4,"{'name': 'G', 'type': 'criterion'}","{'name': 'clinically significant distress', 't...","{'name': 'impairment in social, occupational, ...",None,None,None,c2f184c7-6fc4-4523-98e2-d73d17c97efd,None,None,None,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60,"{'name': 'Specific learning disorder', 'type':...","{'name': 'reading', 'type': 'specifier'}","{'name': 'mathematics', 'type': 'specifier'}","{'name': 'written expression', 'type': 'specif...",None,None,ac718203-696d-44f0-9968-6b9136577bd0,None,None,None,...,"{'subject': 'Specific learning disorder', 'pre...",None,None,None,None,None,None,None,None,None
61,"{'name': '3', 'type': 'criterion'}","{'name': 'Persistent, distorted cognitions abo...","{'name': 'blame himself/herself or others', 't...","{'name': 'traumatic event(s)', 'type': 'concept'}",None,None,cfa80b70-aec2-4879-a959-70085811a2e4,None,None,None,...,"{'subject': 'Persistent, distorted cognitions ...",None,None,None,None,None,None,None,None,None
62,"{'name': 'F17.209', 'type': 'code'}",{'name': 'Unspecified Tobacco-Related Disorder...,None,None,None,None,466f6751-7074-4917-bcbd-383be49e4dc8,None,None,None,...,None,None,None,None,None,None,None,None,None,None
63,"{'name': 'Other specified, or unspecified, tic...","{'name': 'tics', 'type': 'symptom'}","{'name': 'criteria', 'type': 'criterion'}",None,None,None,ac618360-b1da-494d-98ae-cf1c2739b66c,None,None,None,...,None,None,None,None,None,None,None,None,None,None


In [128]:
df.columns

Index(['0_entity', '1_entity', '2_entity', '3_entity', '4_entity', '5_entity',
       '6_entity', '7_entity', '8_entity', 'node_id',
       ...
       '33_relation', '34_relation', '35_relation', '36_relation',
       '37_relation', '38_relation', '39_relation', '40_relation',
       '41_relation', '42_relation'],
      dtype='str', length=105)

In [138]:
pd.concat([df, df2]).columns

Index(['0_entity', '1_entity', '2_entity', '3_entity', '4_entity', '5_entity',
       '6_entity', '7_entity', '8_entity', 'node_id',
       ...
       '34_relation', '35_relation', '36_relation', '37_relation',
       '38_relation', '39_relation', '40_relation', '41_relation',
       '42_relation', '12'],
      dtype='str', length=106)

In [140]:
df3 = pd.concat([df,df2])
df3.to_json("books/dsm5-KG/temp.jsonl", lines=True, orient='records')

In [142]:
df = pd.read_json('books/dsm-tree/dsm5_selected_chapters_tree.jsonl', lines=True)
df

,node_id,parent_id,type,heading,content,page,node_type_in_tree,book_name,section,_path
0,e5cf362b-2ff0-4c7e-be69-05bd426e1938,NaN,root,Chapter 1,,34,parent,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders
1,30b04a24-ef1e-4a70-b522-9f144b51c721,e5cf362b-2ff0-4c7e-be69-05bd426e1938,heading,Neurodevelopmental Disorders,Neurodevelopmental Disorders,34,parent,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders
2,88eb8056-30b0-4204-9ec4-75b0bd74e170,30b04a24-ef1e-4a70-b522-9f144b51c721,paragraph,NaN,"Prior to DSM-5, the name of this chapter was e...",34,leaf,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders -> Neu...
3,144dd110-20d1-49e5-b628-1d223e049be7,30b04a24-ef1e-4a70-b522-9f144b51c721,heading,Quick Guide to the Neurodevelopmental Disorders,Quick Guide to the Neurodevelopmental Disorders,34,parent,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders -> Neu...
4,31e10ee3-d2a9-4ab1-ae80-9589f1e33b7c,144dd110-20d1-49e5-b628-1d223e049be7,paragraph,NaN,"In every Quick Guide, the page number followin...",34,leaf,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders -> Neu...
...,...,...,...,...,...,...,...,...,...,...
17667,466f6751-7074-4917-bcbd-383be49e4dc8,ca0549e3-b27f-4157-b22b-481b60a734c4,heading,F17.209 Unspecified Tobacco-Related Disorder,F17.209 Unspecified Tobacco-Related Disorder,427,leaf,DSM5_ME,Chapter 15: Substance-Related and Addictive Di...,Chapter 15: Substance-Related and Addictive Di...
17668,42bc762f-1783-4c53-87af-15aedd0ef140,ca0549e3-b27f-4157-b22b-481b60a734c4,heading,Other (or Unknown) Substance-Related Disorders,Other (or Unknown) Substance-Related Disorders,427,parent,DSM5_ME,Chapter 15: Substance-Related and Addictive Di...,Chapter 15: Substance-Related and Addictive Di...
17669,45e6309f-9736-46f4-9a2e-79c21eac319b,42bc762f-1783-4c53-87af-15aedd0ef140,paragraph,NaN,The category of other (or unknown) substance-r...,427,leaf,DSM5_ME,Chapter 15: Substance-Related and Addictive Di...,Chapter 15: Substance-Related and Addictive Di...
17670,24f93c36-3649-416a-adae-6681d3d3347c,42bc762f-1783-4c53-87af-15aedd0ef140,paragraph,NaN,Here are some examples of the substances that ...,427,parent,DSM5_ME,Chapter 15: Substance-Related and Addictive Di...,Chapter 15: Substance-Related and Addictive Di...


In [141]:
pd.read_json("books/dsm5-KG/temp.jsonl", lines=True)

,0_entity,1_entity,2_entity,3_entity,4_entity,5_entity,6_entity,7_entity,8_entity,node_id,...,34_relation,35_relation,36_relation,37_relation,38_relation,39_relation,40_relation,41_relation,42_relation,12
0,"{'name': 'Body dysmorphic disorder', 'type': '...","{'name': 'appearance-related preoccupations', ...","{'name': 'repetitive behaviors', 'type': 'symp...","{'name': 'clinically significant distress', 't...","{'name': 'impairment in functioning', 'type': ...","{'name': 'Physical defects', 'type': 'concept'}","{'name': 'skin picking', 'type': 'symptom'}","{'name': 'skin lesions', 'type': 'symptom'}","{'name': 'scarring', 'type': 'symptom'}",7cba5858-4550-43d3-9ffa-eb8b59cc2eed,...,None,None,None,None,None,None,None,None,None,None
1,"{'name': 'committing suicide', 'type': 'symptom'}","{'name': 'B', 'type': 'criterion'}","{'name': 'Mixed symptoms', 'type': 'symptom'}","{'name': 'C', 'type': 'criterion'}","{'name': 'mania', 'type': 'disorder'}","{'name': 'depression', 'type': 'disorder'}","{'name': 'manic episode, with mixed features',...","{'name': 'marked impairment', 'type': 'criteri...","{'name': 'clinical severity', 'type': 'criteri...",0ec02fd1-7cc3-49d1-839e-714ce38e563c,...,None,None,None,None,None,None,None,None,None,None
2,"{'name': 'Russell', 'type': 'patient'}","{'name': 'sweaty', 'type': 'symptom'}","{'name': 'chills', 'type': 'symptom'}","{'name': 'headache', 'type': 'symptom'}","{'name': 'cramping abdominal pain', 'type': 's...","{'name': 'flu', 'type': 'disorder'}","{'name': 'agitated complaint', 'type': 'symptom'}","{'name': 'no fever', 'type': 'symptom'}","{'name': 'pothead coming unglued', 'type': 'co...",4c279985-76de-41a4-88fa-6500fca649d0,...,None,None,None,None,None,None,None,None,None,None
3,"{'name': 'male', 'type': 'concept'}","{'name': 'childhood', 'type': 'duration'}","{'name': 'adolescence', 'type': 'duration'}",{'name': 'interest in various aspects of fire'...,"{'name': 'false alarms', 'type': 'symptom'}","{'name': 'spectators at fires', 'type': 'sympt...",{'name': 'collect the apparatus used by firefi...,"{'name': 'volunteer firefighters', 'type': 'co...","{'name': 'Pyromania', 'type': 'disorder'}",a2d9bb4f-9efc-4914-bb78-433dec1d2d5d,...,None,None,None,None,None,None,None,None,None,None
4,"{'name': 'phencyclidine intoxication', 'type':...",{'name': 'phencyclidine-induced mental disorde...,"{'name': 'heavy use', 'type': 'concept'}","{'name': 'phencyclidine use disorder', 'type':...",None,None,None,None,None,22b0bd97-0d32-4af9-932f-ba69b17cce4c,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11033,"{'name': 'Specific learning disorder', 'type':...","{'name': 'reading', 'type': 'specifier'}","{'name': 'mathematics', 'type': 'specifier'}","{'name': 'written expression', 'type': 'specif...",None,None,None,None,None,ac718203-696d-44f0-9968-6b9136577bd0,...,None,None,None,None,None,None,None,None,None,None
11034,"{'name': '3', 'type': 'criterion'}","{'name': 'Persistent, distorted cognitions abo...","{'name': 'blame himself/herself or others', 't...","{'name': 'traumatic event(s)', 'type': 'concept'}",None,None,None,None,None,cfa80b70-aec2-4879-a959-70085811a2e4,...,None,None,None,None,None,None,None,None,None,None
11035,"{'name': 'F17.209', 'type': 'code'}",{'name': 'Unspecified Tobacco-Related Disorder...,None,None,None,None,None,None,None,466f6751-7074-4917-bcbd-383be49e4dc8,...,None,None,None,None,None,None,None,None,None,None
11036,"{'name': 'Other specified, or unspecified, tic...","{'name': 'tics', 'type': 'symptom'}","{'name': 'criteria', 'type': 'criterion'}",None,None,None,None,None,None,ac618360-b1da-494d-98ae-cf1c2739b66c,...,None,None,None,None,None,None,None,None,None,None


In [119]:
df1 = pd.concat([df,df2], axis=1)
df1

,0_entity,1_entity,2_entity,3_entity,4_entity,5_entity,6_entity,7_entity,8_entity,node_id,...,2_relation,3_relation,4_relation,5_relation,6_relation,7_relation,8_relation,9_relation,10_relation,11_relation
0,"{'name': 'Body dysmorphic disorder', 'type': '...","{'name': 'appearance-related preoccupations', ...","{'name': 'repetitive behaviors', 'type': 'symp...","{'name': 'clinically significant distress', 't...","{'name': 'impairment in functioning', 'type': ...","{'name': 'Physical defects', 'type': 'concept'}","{'name': 'skin picking', 'type': 'symptom'}","{'name': 'skin lesions', 'type': 'symptom'}","{'name': 'scarring', 'type': 'symptom'}",7cba5858-4550-43d3-9ffa-eb8b59cc2eed,...,"{'subject': 'unfounded fear of a serious, ofte...","{'subject': 'unfounded fear of a serious, ofte...","{'subject': 'Illness anxiety disorder', 'predi...",None,None,None,None,None,None,None
1,"{'name': 'committing suicide', 'type': 'symptom'}","{'name': 'B', 'type': 'criterion'}","{'name': 'Mixed symptoms', 'type': 'symptom'}","{'name': 'C', 'type': 'criterion'}","{'name': 'mania', 'type': 'disorder'}","{'name': 'depression', 'type': 'disorder'}","{'name': 'manic episode, with mixed features',...","{'name': 'marked impairment', 'type': 'criteri...","{'name': 'clinical severity', 'type': 'criteri...",0ec02fd1-7cc3-49d1-839e-714ce38e563c,...,{'subject': 'childhood-onset fluency disorder'...,"{'subject': 'Tourette’s disorder', 'predicate'...","{'subject': 'Vocal tics', 'predicate': 'distin...","{'subject': 'repetitive vocalizations', 'predi...",None,None,None,None,None,None
2,"{'name': 'Russell', 'type': 'patient'}","{'name': 'sweaty', 'type': 'symptom'}","{'name': 'chills', 'type': 'symptom'}","{'name': 'headache', 'type': 'symptom'}","{'name': 'cramping abdominal pain', 'type': 's...","{'name': 'flu', 'type': 'disorder'}","{'name': 'agitated complaint', 'type': 'symptom'}","{'name': 'no fever', 'type': 'symptom'}","{'name': 'pothead coming unglued', 'type': 'co...",4c279985-76de-41a4-88fa-6500fca649d0,...,None,None,None,None,None,None,None,None,None,None
3,"{'name': 'male', 'type': 'concept'}","{'name': 'childhood', 'type': 'duration'}","{'name': 'adolescence', 'type': 'duration'}",{'name': 'interest in various aspects of fire'...,"{'name': 'false alarms', 'type': 'symptom'}","{'name': 'spectators at fires', 'type': 'sympt...",{'name': 'collect the apparatus used by firefi...,"{'name': 'volunteer firefighters', 'type': 'co...","{'name': 'Pyromania', 'type': 'disorder'}",a2d9bb4f-9efc-4914-bb78-433dec1d2d5d,...,None,None,None,None,None,None,None,None,None,None
4,"{'name': 'phencyclidine intoxication', 'type':...",{'name': 'phencyclidine-induced mental disorde...,"{'name': 'heavy use', 'type': 'concept'}","{'name': 'phencyclidine use disorder', 'type':...",None,None,None,None,None,22b0bd97-0d32-4af9-932f-ba69b17cce4c,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10968,"{'name': 'Cannabis', 'type': 'concept'}",{'name': 'taken in larger amounts or over a lo...,None,None,None,None,None,None,None,f544dbb8-3496-4dd1-8b18-a6e3a3e85839,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10969,"{'name': 'Severity', 'type': 'specifier'}","{'name': 'psychosis', 'type': 'concept'}","{'name': 'delusions', 'type': 'symptom'}","{'name': 'hallucinations', 'type': 'symptom'}","{'name': 'disorganized speech', 'type': 'sympt...","{'name': 'abnormal psychomotor behavior', 'typ...","{'name': 'negative symptoms', 'type': 'symptom'}","{'name': 'last 7 days', 'type': 'duration'}","{'name': 'delusional disorder', 'type': 'disor...",72c5a9e9-0aaa-4ca2-ad83-4e63c7821436,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10970,{'name': 'substance/medication-induced bipolar...,"{'name': 'bipolar I disorder', 'type': 'disord...","{'name': 'substance', 'type': 'concept'}","{'name': 'stimulants', 'type': 'concept'}","{'name': 'phencyclidine', 'type': 'concept'}","{'name': 'medication', 'type': 'conce

In [123]:
df1['node_id']

,node_id,node_id
0,7cba5858-4550-43d3-9ffa-eb8b59cc2eed,1c42e031-05d0-482c-bd07-54e4cbaa7dbe
1,0ec02fd1-7cc3-49d1-839e-714ce38e563c,f4b7f9dd-2f0d-4c5a-a249-2dbd45c3418d
2,4c279985-76de-41a4-88fa-6500fca649d0,15d97c53-1d5d-49e5-abad-d4ee6b910fd9
3,a2d9bb4f-9efc-4914-bb78-433dec1d2d5d,a8a1244a-801a-4e19-b5d3-edacaf5b30aa
4,22b0bd97-0d32-4af9-932f-ba69b17cce4c,c2f184c7-6fc4-4523-98e2-d73d17c97efd
...,...,...
10968,f544dbb8-3496-4dd1-8b18-a6e3a3e85839,NaN
10969,72c5a9e9-0aaa-4ca2-ad83-4e63c7821436,NaN
10970,b71ddd68-e207-43d1-bef0-74249f3aebdd,NaN
10971,ccdd383b-5ff1-48b8-947b-d35fda04dfbf,NaN


In [5]:
df_junks = pd.read_json("books/dsm5-KG/dsm5-bad_nodes.jsonl",lines=True)
df_junks

,index,node_id
0,0,01d2489a-13e0-4fb1-aeb9-4f88948d22c9
1,0,90787552-c35e-41eb-bc59-91075b9150b8
2,0,13c2522d-5a31-4c07-a62f-11835dc4723c
3,0,1e4bc0d2-729c-4d3c-99aa-d10f849536b2
4,0,5ad1957a-aaa3-4026-a891-aa921a8661e2
...,...,...
839,0,7e7307c3-a849-4bb6-a9dd-6bd9b1520fdb
840,0,c5989bb2-8bfa-4e7c-8c6f-5073801bd144
841,0,3f51c930-cdd3-4636-903a-fd53f4db1389
842,0,d1aee87b-27ae-406a-8bcb-5a9e338aecea


In [93]:
df_all = pd.read_json("books/dsm-tree/dsm5_selected_chapters_tree.jsonl",lines=True)
df_all

,node_id,parent_id,type,heading,content,page,node_type_in_tree,book_name,section,_path
0,e5cf362b-2ff0-4c7e-be69-05bd426e1938,NaN,root,Chapter 1,,34,parent,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders
1,30b04a24-ef1e-4a70-b522-9f144b51c721,e5cf362b-2ff0-4c7e-be69-05bd426e1938,heading,Neurodevelopmental Disorders,Neurodevelopmental Disorders,34,parent,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders
2,88eb8056-30b0-4204-9ec4-75b0bd74e170,30b04a24-ef1e-4a70-b522-9f144b51c721,paragraph,NaN,"Prior to DSM-5, the name of this chapter was e...",34,leaf,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders -> Neu...
3,144dd110-20d1-49e5-b628-1d223e049be7,30b04a24-ef1e-4a70-b522-9f144b51c721,heading,Quick Guide to the Neurodevelopmental Disorders,Quick Guide to the Neurodevelopmental Disorders,34,parent,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders -> Neu...
4,31e10ee3-d2a9-4ab1-ae80-9589f1e33b7c,144dd110-20d1-49e5-b628-1d223e049be7,paragraph,NaN,"In every Quick Guide, the page number followin...",34,leaf,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders -> Neu...
...,...,...,...,...,...,...,...,...,...,...
17570,0e5e1d9b-60e0-483d-be9a-5b0ece32f4ea,4b22b5f3-56d9-4b57-baf0-2fdea73c1bdd,list_item,NaN,❑ Not\n\npresent,1119,leaf,DSM5_TR,Assessment Measures,Assessment Measures -> VIII. Mania
17571,01629128-abc9-49b0-b12f-b28fd50b5624,4b22b5f3-56d9-4b57-baf0-2fdea73c1bdd,list_item,NaN,"❑ Equivocal (occasional\n\nelevated, expansive...",1119,leaf,DSM5_TR,Assessment Measures,Assessment Measures -> VIII. Mania
17572,1beb01c6-3457-440d-a121-6951586a33a0,4b22b5f3-56d9-4b57-baf0-2fdea73c1bdd,list_item,NaN,"❑ Present, but\n\nmild (frequent periods of so...",1119,leaf,DSM5_TR,Assessment Measures,Assessment Measures -> VIII. Mania
17573,d1757e9a-b3d2-40ba-b497-c74c3b71e7c9,4b22b5f3-56d9-4b57-baf0-2fdea73c1bdd,list_item,NaN,❑ Present and\n\nmoderate (frequent periods of...,1119,leaf,DSM5_TR,Assessment Measures,Assessment Measures -> VIII. Mania


In [97]:
df

,node_id,parent_id,type,heading,content,page,node_type_in_tree,book_name,section,_path
0,9be36887-a719-4869-9487-fbb517bf6079,NaN,root,NaN,,151,parent,DSM5_TR,Neurodevelopmental Disorders,Neurodevelopmental Disorders
1,ca17614b-b965-49dc-b391-38badb62b886,9be36887-a719-4869-9487-fbb517bf6079,heading,Sensory deficits.,Sensory deficits.,151,parent,DSM5_TR,Neurodevelopmental Disorders,Neurodevelopmental Disorders
2,d9f56e7c-d243-4d42-a8ba-c4427d0de208,ca17614b-b965-49dc-b391-38badb62b886,paragraph,NaN,Dysfluencies of speech may be associated with ...,151,leaf,DSM5_TR,Neurodevelopmental Disorders,Neurodevelopmental Disorders -> Sensory deficits.
3,2fa7dd37-5da6-450d-a5c2-2ef9b29cbe72,9be36887-a719-4869-9487-fbb517bf6079,heading,Normal speech dysfluencies.,Normal speech dysfluencies.,151,parent,DSM5_TR,Neurodevelopmental Disorders,Neurodevelopmental Disorders
4,b9de616e-5b37-408e-98ca-cd61abbb4028,2fa7dd37-5da6-450d-a5c2-2ef9b29cbe72,paragraph,NaN,The disorder must be distinguished from normal...,151,leaf,DSM5_TR,Neurodevelopmental Disorders,Neurodevelopmental Disorders -> Normal speech ...
...,...,...,...,...,...,...,...,...,...,...
92,466f6751-7074-4917-bcbd-383be49e4dc8,ca0549e3-b27f-4157-b22b-481b60a734c4,heading,F17.209 Unspecified Tobacco-Related Disorder,F17.209 Unspecified Tobacco-Related Disorder,427,leaf,DSM5_ME,Chapter 15: Substance-Related and Addictive Di...,Chapter 15: Substance-Related and Addictive Di...
93,42bc762f-1783-4c53-87af-15aedd0ef140,ca0549e3-b27f-4157-b22b-481b60a734c4,heading,Other (or Unknown) Substance-Related Disorders,Other (or Unknown) Substance-Related Disorders,427,parent,DSM5_ME,Chapter 15: Substance-Related and Addictive Di...,Chapter 15: Substance-Related and Addictive Di...
94,45e6309f-9736-46f4-9a2e-79c21eac319b,42bc762f-1783-4c53-87af-15aedd0ef140,paragraph,NaN,The category of other (or unknown) substance-r...,427,leaf,DSM5_ME,Chapter 15: Substance-Related and Addictive Di...,Chapter 15: Substance-Related and Addictive Di...
95,24f93c36-3649-416a-adae-6681d3d3347c,42bc762f-1783-4c53-87af-15aedd0ef140,paragraph,NaN,Here are some examples of the substances that ...,427,parent,DSM5_ME,Chapter 15: Substance-Related and Addictive Di...,Chapter 15: Substance-Related and Addictive Di...


In [99]:
pd.concat([df_all, df]).to_json("books/dsm-tree/dsm5_selected_chapters_tree.jsonl", orient='records', lines=True)

In [ ]:
df_all[(df_all['page']==142) & (df_all['book_name']=='DSM5_ME')]

,node_id,parent_id,type,heading,content,page,node_type_in_tree,book_name,section,_path
796,a6bd4b97-4400-43db-98ce-7353e330bc21,NaN,root,NaN,,142,parent,DSM5_ME,Chapter 3: Mood Disorders,Chapter 3: Mood Disorders
797,da1ca34e-99dd-4187-a002-4e9152dcdb24,a6bd4b97-4400-43db-98ce-7353e330bc21,case_study,NaN,compensate for this “inherent second-rateness....,142,leaf,DSM5_ME,Chapter 3: Mood Disorders,Chapter 3: Mood Disorders
798,a5819632-c4f8-45ca-8758-8be1698dbcb3,a6bd4b97-4400-43db-98ce-7353e330bc21,heading,Evaluation of Noah Sanders,Evaluation of Noah Sanders,142,parent,DSM5_ME,Chapter 3: Mood Disorders,Chapter 3: Mood Disorders
799,d4fa37c4-b825-45c5-8187-d3be29171fbe,a5819632-c4f8-45ca-8758-8be1698dbcb3,paragraph,NaN,"For most of his adult life, Noah has had depre...",142,leaf,DSM5_ME,Chapter 3: Mood Disorders,Chapter 3: Mood Disorders -> Evaluation of Noa...


In [9]:
bad_pages = pd.merge(df_all, df_junks)
bad_pages

,node_id,parent_id,type,heading,content,page,node_type_in_tree,book_name,section,_path,index
0,33d3dfc4-de21-4972-b808-41080469ec6a,f8a3e26e-e6c9-4923-ada3-8df8759f23d4,paragraph,NaN,The many causes of IDD include genetic abnorma...,37,leaf,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders,0
1,3bfe0f99-2a8e-485c-9e4a-e9a083a9fe52,c4d5b51c-07fe-4216-a31b-ad7f9867822f,paragraph,NaN,"the interviewer. (However, much of the backgro...",41,leaf,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders,0
2,e1dc8058-028d-4cef-aff2-cb1e3de8e29a,77e4f718-31b0-4ddb-9893-c513091d925b,paragraph,NaN,"phone goes off when she’s giving a lecture, it...",47,leaf,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders,0
3,3d63636e-3c05-43bc-926e-f64430c4dce6,0e83a15b-eb70-4d12-b0f0-42786f709adc,paragraph,NaN,Except for the descriptive specifier “with imp...,63,leaf,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders -> ###...,0
4,b9762416-d285-4dcb-ba5d-fba3a0554087,5a209d35-188b-421b-90a6-2942deb0226c,paragraph,NaN,We must also discriminate hallucinations from ...,71,leaf,DSM5_ME,Chapter 2: Schizophrenia Spectrum and Other Ps...,Chapter 2: Schizophrenia Spectrum and Other Ps...,0
...,...,...,...,...,...,...,...,...,...,...,...
839,f3d51c29-8b3c-4d1e-ae99-e05a21a279a6,9ba32b35-8182-439d-8d10-0c2a37a1cf44,paragraph,NaN,853,1119,leaf,DSM5_TR,Assessment Measures,Assessment Measures,0
840,374485ae-e886-4336-b053-a6afaa0021ac,09583f40-5f76-4f2f-a2e8-ee4a6820ac2a,list_item,NaN,❑ Not\n\npresent,1119,leaf,DSM5_TR,Assessment Measures,Assessment Measures -> V. Negative symptoms (r...,0
841,24f2254d-7a2e-4b4b-89ab-264532eab632,cca20bfe-bcce-49e7-946f-7da58f6232b6,list_item,NaN,❑ Not\n\npresent,1119,leaf,DSM5_TR,Assessment Measures,Assessment Measures -> VI. Impaired\n\ncognition,0
842,2368ccb5-c7aa-44fb-bcd1-b42905b93b56,07c2c9aa-b805-4dfa-9f72-87894ba63078,list_item,NaN,❑ Not\n\npresent,1119,leaf,DSM5_TR,Assessment Measures,Assessment Measures -> VII. Depression,0


In [10]:
bad_pages['con_len'] = bad_pages['content'].apply(lambda x: len(x))
bad_pages

,node_id,parent_id,type,heading,content,page,node_type_in_tree,book_name,section,_path,index,con_len
0,33d3dfc4-de21-4972-b808-41080469ec6a,f8a3e26e-e6c9-4923-ada3-8df8759f23d4,paragraph,NaN,The many causes of IDD include genetic abnorma...,37,leaf,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders,0,164
1,3bfe0f99-2a8e-485c-9e4a-e9a083a9fe52,c4d5b51c-07fe-4216-a31b-ad7f9867822f,paragraph,NaN,"the interviewer. (However, much of the backgro...",41,leaf,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders,0,155
2,e1dc8058-028d-4cef-aff2-cb1e3de8e29a,77e4f718-31b0-4ddb-9893-c513091d925b,paragraph,NaN,"phone goes off when she’s giving a lecture, it...",47,leaf,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders,0,93
3,3d63636e-3c05-43bc-926e-f64430c4dce6,0e83a15b-eb70-4d12-b0f0-42786f709adc,paragraph,NaN,Except for the descriptive specifier “with imp...,63,leaf,DSM5_ME,Chapter 1: Neurodevelopmental Disorders,Chapter 1: Neurodevelopmental Disorders -> ###...,0,171
4,b9762416-d285-4dcb-ba5d-fba3a0554087,5a209d35-188b-421b-90a6-2942deb0226c,paragraph,NaN,We must also discriminate hallucinations from ...,71,leaf,DSM5_ME,Chapter 2: Schizophrenia Spectrum and Other Ps...,Chapter 2: Schizophrenia Spectrum and Other Ps...,0,163
...,...,...,...,...,...,...,...,...,...,...,...,...
839,f3d51c29-8b3c-4d1e-ae99-e05a21a279a6,9ba32b35-8182-439d-8d10-0c2a37a1cf44,paragraph,NaN,853,1119,leaf,DSM5_TR,Assessment Measures,Assessment Measures,0,3
840,374485ae-e886-4336-b053-a6afaa0021ac,09583f40-5f76-4f2f-a2e8-ee4a6820ac2a,list_item,NaN,❑ Not\n\npresent,1119,leaf,DSM5_TR,Assessment Measures,Assessment Measures -> V. Negative symptoms (r...,0,14
841,24f2254d-7a2e-4b4b-89ab-264532eab632,cca20bfe-bcce-49e7-946f-7da58f6232b6,list_item,NaN,❑ Not\n\npresent,1119,leaf,DSM5_TR,Assessment Measures,Assessment Measures -> VI. Impaired\n\ncognition,0,14
842,2368ccb5-c7aa-44fb-bcd1-b42905b93b56,07c2c9aa-b805-4dfa-9f72-87894ba63078,list_item,NaN,❑ Not\n\npresent,1119,leaf,DSM5_TR,Assessment Measures,Assessment Measures -> VII. Depression,0,14


In [22]:
bad_pages.sort_values(by='con_len')

,node_id,parent_id,type,heading,content,page,node_type_in_tree,book_name,section,_path,index,con_len
34,f2988532-7c3d-42a6-b47f-312390512930,NaN,root,NaN,,282,leaf,DSM5_ME,Chapter 9: Feeding and Eating Disorders,Chapter 9: Feeding and Eating Disorders,0,0
245,e9e2e089-f51f-4cab-a18a-cb0ecb62bdd4,7be56295-bfca-4c86-b81a-908bd0151bd8,paragraph,"Multiple episodes, currently in partial remission",,233,leaf,DSM5_TR,Schizophrenia Spectrum and Other Psychotic Dis...,Schizophrenia Spectrum and Other Psychotic Dis...,0,0
246,f4a7cf5e-2854-4c1a-aa58-8c2d50d2f150,7be56295-bfca-4c86-b81a-908bd0151bd8,paragraph,"Multiple episodes, currently in full remission",,233,leaf,DSM5_TR,Schizophrenia Spectrum and Other Psychotic Dis...,Schizophrenia Spectrum and Other Psychotic Dis...,0,0
247,3de91e5d-1403-4449-8f17-65beb87d1834,7be56295-bfca-4c86-b81a-908bd0151bd8,paragraph,Unspecified,,233,leaf,DSM5_TR,Schizophrenia Spectrum and Other Psychotic Dis...,Schizophrenia Spectrum and Other Psychotic Dis...,0,0
100,ad4c2527-ae61-48d4-b585-bff556b397d2,5f290216-6913-4586-94e2-99e84931e68e,paragraph,F19.99 Unspecified Other (or Unknown) Substanc...,,428,leaf,DSM5_ME,Chapter 15: Substance-Related and Addictive Di...,Chapter 15: Substance-Related and Addictive Di...,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
140,246ecc71-db60-4d86-b9bc-4c45592c92a7,b9c1b38d-6598-4303-9f63-713300fc38cd,paragraph,NaN,"Throughout my professional life, I’ve spent a ...",563,leaf,DSM5_ME,Chapter 20: Patients and Diagnoses,Chapter 20: Patients and Diagnoses,0,407
349,a012d4c2-5ee6-42bb-a3e0-113ea4eaf11a,9b54b463-0a24-4db2-8f48-13e25491d191,paragraph,NaN,**Delirium.**\n\ncourse); and evidence in the ...,443,leaf,DSM5_TR,Obsessive-Compulsive and Related Disorders,Obsessive-Compulsive and Related Disorders,0,408
15,e74c2720-4ac2-42d2-9191-80becc5fd315,e94eaf12-3bea-4432-bc19-5c0065aa5425,paragraph,NaN,DSM-IV differentiated between dysthymic disord...,140,leaf,DSM5_ME,Chapter 3: Mood Disorders,Chapter 3: Mood Disorders -> Additional Mood D...,0,412
87,bfbb4184-3e46-4bd9-9ab2-fd3420ed37ea,fc496489-b8b8-4c91-801f-8e3359f99d40,paragraph,NaN,Australia is probably the strictest country of...,393,leaf,DSM5_ME,Chapter 15: Substance-Related and Addictive Di...,Chapter 15: Substance-Related and Addictive Di...,0,419


In [32]:
bad_pages[bad_pages['node_id']=='15272ed1-4cd1-4991-b2bc-7c7fca00b500']

,node_id,parent_id,type,heading,content,page,node_type_in_tree,book_name,section,_path,index,con_len
40,15272ed1-4cd1-4991-b2bc-7c7fca00b500,7c190a8c-ac53-431f-8d44-1ef20520ca8e,paragraph,NaN,"treatment, are highly likely to have relatives...",141,leaf,DSM5_ME,Chapter 3: Mood Disorders,Chapter 3: Mood Disorders,0,141


In [25]:
bad_pages.iloc[15]['content']

'DSM-IV differentiated between dysthymic disorder and chronic major depressive disorder, but research did not validate the distinction. So what DSM-5-TR now calls persistent depressive disorder is a combination of two separate DSM-IV conditions. The current criteria supply some specifiers to indicate the difference. Here’s what’s clear: Patients who have depression that goes on and on tend to respond poorly to'